# FleetMind-LLM｜Phase 4：模型评测与对比

## 阶段目标

在统一的数据和评测规则下，比较规则方法、原始 Qwen3-0.6B 和 LoRA 微调模型，检验微调是否真正改善物流信息提取与交付延误判断。同时检查数据质量、分析错误，并用独立样本观察模型的泛化能力。

Phase 3 已完成 Qwen3-0.6B 的 SFT + 标准 LoRA 微调。已有指令数据为 Train 1,600、Val 200、Test 200。旧 Val 200 已用于模型选择；Test 尚未用于模型评测。


## Step 4.1：评测数据扩展与质量检查

Phase 3 已使用 1,600 条训练样本、200 条验证样本完成 Qwen3-0.6B 的 SFT + 标准 LoRA 微调。本步骤沿用已有的业务记录划分，计划将验证集扩展至 1,000 条、测试集扩展至 2,000 条，并保留原有样本。

| 数据集 | 当前指令样本数 | 本阶段目标 | 用途 |
| --- | ---: | ---: | --- |
| Train | 1,600 | 暂不扩展 | 已完成的 Phase 3 模型训练 |
| Val | 200 | 1,000 | 模型比较、错误分析与选择 |
| Test | 200 | 2,000 | 最终独立评测，扩展后封存 |

**重点检查**：

1. Train、Val、Test 中的业务记录不能重叠；同一记录的 Task A 和 Task B 样本应属于同一数据集。
2. Task A 的答案应是有效 JSON，字段、类型和缺失值应与原始记录一致。
3. Task B 的标签由计划交付时间和实际完成时间直接比较得出；实际时间**晚于**计划时间才标为 `delayed`。
4. 检查类别分布、重复样本，以及输入中是否出现直接暴露标准答案的字段。
5. 记录抽样规则、随机种子和数据版本，保证后续实验可以复现。

**评测边界**：旧 Val 200 已用于 Phase 3 模型选择，因此扩展后的 Val 仍是开发集。Test 在最终模型和评测方法确定前保持封存。

In [ ]:
from google.colab import drive

# 将你的 Google Drive 挂载到当前 Colab 运行环境
drive.mount("/content/drive")

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

# 打开本机文件选择窗口，选择桌面上的正式版数据压缩包
uploaded = files.upload()

# 确认上传的文件名、大小，并查看压缩包内有哪些文件
for name in uploaded:
    path = Path("/content") / name
    print(f"\n已上传：{path}")
    print(f"大小：{path.stat().st_size / 1024**2:.2f} MB")

    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as archive:
            print("压缩包内容：")
            for item in archive.infolist():
                print(f"  {item.filename}  ({item.file_size / 1024**2:.2f} MB)")
    else:
        print("该文件不是 ZIP 压缩包。")

现在先核对样本格式和原始划分，不读取 Test 的题目或答案，也不改动压缩包。

In [ ]:
import json
import csv
import hashlib
import zipfile
from collections import Counter
from pathlib import Path

zip_path = Path("/content/FleetMind_LLM_Phase1_Dataset.zip")
prefix = "fleetmind_phase1_v1/"

with zipfile.ZipFile(zip_path) as archive:
    # 读取数据说明，确认原数据集的生成与划分信息
    info = json.loads(archive.read(prefix + "dataset_info.json"))
    print("===== dataset_info.json =====")
    print(json.dumps(info, ensure_ascii=False, indent=2)[:12000])

    # 核对三份指令文件的行数；只展示 Train 的一个样本结构
    print("\n===== JSONL 数量与结构 =====")
    for split in ("train", "val", "test"):
        name = prefix + f"{split}.jsonl"
        lines = archive.read(name).decode("utf-8").splitlines()
        print(f"{split}: {len(lines)} 条")

        if split == "train" and lines:
            example = json.loads(lines[0])
            print("Train 样本字段：", list(example.keys()))
            print("Train 首条样本：")
            print(json.dumps(example, ensure_ascii=False, indent=2)[:2500])

    # 查看划分表的列名、数量和前几行，确认业务记录 ID 的字段
    print("\n===== split_assignments.csv =====")
    with archive.open(prefix + "split_assignments.csv") as file:
        reader = csv.DictReader(
            (line.decode("utf-8-sig") for line in file)
        )
        print("列名：", reader.fieldnames)
        first_rows = []
        total = 0
        for row in reader:
            total += 1
            if len(first_rows) < 5:
                first_rows.append(row)

        print("总行数：", total)
        print("前 5 行：")
        for row in first_rows:
            print(row)

# 记录压缩包版本，便于以后核对是否使用同一份数据
sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
print("\n数据包 SHA-256：", sha256)

**防泄漏初查**：下面这格只读取样本 ID 和划分信息，不查看 Test 的题目或答案

In [ ]:
import csv
import json
import zipfile
from collections import Counter, defaultdict

zip_path = "/content/FleetMind_LLM_Phase1_Dataset.zip"
prefix = "fleetmind_phase1_v1/"

with zipfile.ZipFile(zip_path) as archive:
    # 读取 84,776 条业务记录所属的原始划分
    with archive.open(prefix + "split_assignments.csv") as file:
        rows = csv.DictReader(
            line.decode("utf-8-sig") for line in file
        )
        assignments = {row["load_id"]: row for row in rows}

    samples = {}
    for split in ("train", "val", "test"):
        # 只读取样本元数据，不打印题目和标准答案
        raw_lines = archive.read(
            prefix + f"{split}.jsonl"
        ).decode("utf-8").splitlines()
        samples[split] = [json.loads(line) for line in raw_lines]

# 分别检查每个集合的数量、A/B 配对、ID 重复和原始划分
load_ids_by_split = {}
all_sample_ids = []

for split, items in samples.items():
    sample_ids = [x["sample_id"] for x in items]
    all_sample_ids.extend(sample_ids)

    tasks_by_load = defaultdict(list)
    wrong_assignment = []
    wrong_trip = []
    wrong_sample_id = []
    wrong_split_field = []

    for x in items:
        load_id = x["source_load_id"]
        trip_id = x["source_trip_id"]
        task = x["task"]
        tasks_by_load[load_id].append(task)

        assigned = assignments.get(load_id)
        if assigned is None or assigned["split"] != split:
            wrong_assignment.append(load_id)
        elif assigned["trip_id"] != trip_id:
            wrong_trip.append(load_id)

        if x["sample_id"] != f"{load_id}_{task}":
            wrong_sample_id.append(x["sample_id"])
        if x["split"] != split:
            wrong_split_field.append(x["sample_id"])

    load_ids_by_split[split] = set(tasks_by_load)
    pair_issues = [
        load_id for load_id, tasks in tasks_by_load.items()
        if sorted(tasks) != ["A", "B"]
    ]

    print(f"\n===== {split} =====")
    print("指令样本数：", len(items))
    print("独立业务记录数：", len(tasks_by_load))
    print("任务数量：", dict(Counter(x["task"] for x in items)))
    print("重复 sample_id：", len(sample_ids) - len(set(sample_ids)))
    print("A/B 配对异常：", len(pair_issues))
    print("与原划分表不符：", len(wrong_assignment))
    print("trip_id 不符：", len(wrong_trip))
    print("sample_id 格式异常：", len(wrong_sample_id))
    print("样本 split 字段不符：", len(wrong_split_field))

# 检查不同集合是否包含同一条业务记录
print("\n===== 跨集合检查 =====")
for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    overlap = load_ids_by_split[left] & load_ids_by_split[right]
    print(f"{left} / {right} 业务记录重叠：{len(overlap)}")

print("全部集合重复 sample_id：",
      len(all_sample_ids) - len(set(all_sample_ids)))
print("划分表各组业务记录数：",
      dict(Counter(row["split"] for row in assignments.values())))

结果显示 A/B 配对、样本 ID 和跨集合重叠检查都正常。Val 的两个“200 条不符”很可能只是命名差异：划分表写 validation，JSONL 文件名写 val。**下面这格专门复核 Val**；先修正检查口径，不能把它记作数据错误。

In [ ]:
from collections import Counter

# 文件名 val 对应划分表中的 validation
expected_assignment = {"train": "train", "val": "validation", "test": "test"}

for split, items in samples.items():
    wrong_assignment = []
    wrong_split_field = []

    for x in items:
        load_id = x["source_load_id"]
        assigned = assignments.get(load_id)

        # 用划分表的正式名称检查业务记录归属
        if assigned is None or assigned["split"] != expected_assignment[split]:
            wrong_assignment.append(x["sample_id"])

        # 先统计 JSONL 内实际使用的 split 名称，避免误判命名别名
        if x["split"] not in {split, expected_assignment[split]}:
            wrong_split_field.append(x["sample_id"])

    print(f"\n{split}")
    print("样本内 split 字段取值：",
          dict(Counter(x["split"] for x in items)))
    print("与划分表不符：", len(wrong_assignment))
    print("无法识别的 split 字段：", len(wrong_split_field))

### 4.1.1 现有样本与数据划分检查

正式数据包包含 Train 1,600、Val 200、Test 200 条指令样本。每条业务记录分别生成一条 Task A 和一条 Task B 样本，因此对应 800、100、100 条独立业务记录。

| 检查项 | 结果 |
| --- | --- |
| A/B 样本配对异常 | 0 |
| 重复 `sample_id` | 0 |
| 与原始业务记录划分不符 | 0 |
| Train / Val / Test 业务记录交叉 | 0 |
| 运单与行程 ID 对应错误 | 0 |

**方法与作用**：使用业务记录 ID、样本 ID 和原划分表进行逐条核对，确认当前指令数据没有发现记录级交叉或配对错误。Val 文件名为 `val.jsonl`，其中的 `split` 字段使用 `validation`；两者指向同一划分，已按该命名规则复核。

**范围说明**：本次只检查现有样本的划分与身份信息。Task A 字段标签、Task B 时间标签和新增样本的质量仍需单独检查。

接下来从桌面上传 Phase 1 清洗合并后的业务明细表，即包含 load_id、`truck_id`、城市、收入、耗油量、计划与实际交付时间的 CSV/Parquet/ZIP 文件。`split_assignments.csv` 只有运单归属，没有生成新题目所需的业务字段。

In [ ]:
from google.colab import files

# 从桌面选择 Phase 1 清洗合并后的业务明细文件
uploaded_business = files.upload()

# 查看实际文件名和大小，下一步据此读取字段
for name, content in uploaded_business.items():
    print(f"已上传：/content/{name}，大小：{len(content) / 1024**2:.2f} MB")

### 4.1.2 原始业务表还原与旧标签核对

**目的**：从原始业务数据中找回生成指令样本所需的字段，并确认字段映射与 Phase 1 的正式数据一致。只有核对通过，才用剩余业务记录扩展验证集和测试集。

本步骤读取 `loads`、`trips`、`routes` 和 `delivery_events` 四张表，按运单、行程和路线 ID 合并，再用既有的 `split_assignments.csv` 保留原来的 Train / Validation / Test 归属。交付时间取自 `Delivery` 事件。

| 技术 | 作用 |
| --- | --- |
| Pandas 表连接及 `validate` | 合并业务字段，并检查连接关系是否意外产生重复行 |
| 原划分表回查 | 保持 Phase 1 的记录归属，避免重新划分造成数据泄漏 |
| 旧 Train 标签逐字段对照 | 检查收入、耗油量、城市、车辆及时间等字段是否与原始数据一致 |

**检查标准**：合并后应有 84,776 条已划分业务记录，其中 Train 67,820、Validation 8,478、Test 8,478；旧 Train 中实际展示的 Task A 字段应与源记录一致。

> 状态：等待代码运行。本节只说明方法与预期，实际结果将在核对输出后补充。

In [ ]:
import json
import zipfile
import pandas as pd
from collections import Counter

raw_zip = "/content/Fleetmind_dataset.zip"
data_zip = "/content/FleetMind_LLM_Phase1_Dataset.zip"

# 从 ZIP 直接读取四张原始业务表，不修改原文件
with zipfile.ZipFile(raw_zip) as archive:
    base = "Fleetmind_dataset/"
    loads = pd.read_csv(archive.open(base + "loads.csv"))
    trips = pd.read_csv(archive.open(base + "trips.csv"))
    routes = pd.read_csv(archive.open(base + "routes.csv"))
    events = pd.read_csv(archive.open(base + "delivery_events.csv"))

print("原始表行数：", {
    "loads": len(loads),
    "trips": len(trips),
    "routes": len(routes),
    "delivery_events": len(events),
})
print("事件类型：", events["event_type"].value_counts().to_dict())

# Task B 使用交付事件的计划时间和实际时间
delivery = events.loc[
    events["event_type"] == "Delivery",
    ["load_id", "trip_id", "scheduled_datetime", "actual_datetime"]
].copy()

# 按业务键合并；validate 用于阻止意外的一对多合并
business = (
    loads.merge(trips, on="load_id", how="inner", validate="one_to_one")
         .merge(routes, on="route_id", how="inner", validate="many_to_one")
         .merge(delivery, on=["load_id", "trip_id"],
                how="inner", validate="one_to_one")
)

# 只保留 Phase 1 已接受、且已固定 Train/Validation/Test 归属的记录
with zipfile.ZipFile(data_zip) as archive:
    split_table = pd.read_csv(
        archive.open("fleetmind_phase1_v1/split_assignments.csv")
    )

business = business.merge(
    split_table,
    on=["load_id", "trip_id"],
    how="inner",
    validate="one_to_one"
)

print("\n合并后已划分业务记录：", len(business))
print("各组记录数：", business["split"].value_counts().to_dict())
print("重复 load_id：", business["load_id"].duplicated().sum())

# 将源字段映射到 Task A 的标准答案字段
field_map = {
    "load_id": "load_id",
    "truck_id": "truck_id",
    "origin_city": "origin_city",
    "destination_city": "destination_city",
    "revenue_usd": "revenue",
    "fuel_used_gallons": "fuel_gallons_used",
    "scheduled_delivery": "scheduled_datetime",
    "actual_delivery": "actual_datetime",
}

business_by_id = business.set_index("load_id", drop=False)

# 用旧 Train 的 Task A 标签核对映射；仅比较输入实际展示的字段
# 缺失字段在旧标签里是 null，不拿原始表的隐藏值比较
mismatches = Counter()
checked = 0

for sample in samples["train"]:
    if sample["task"] != "A":
        continue

    source = business_by_id.loc[sample["source_load_id"]]
    answer = json.loads(sample["messages"][-1]["content"])
    checked += 1

    for answer_field in sample["shown_fields"]:
        source_value = source[field_map[answer_field]]
        expected = answer[answer_field]

        if isinstance(expected, (int, float)):
            equal = abs(float(source_value) - float(expected)) < 1e-6
        else:
            equal = str(source_value) == str(expected)

        if not equal:
            mismatches[answer_field] += 1

print("\n旧 Train Task A 对照数：", checked)
print("各字段不一致次数：", dict(mismatches))

**实测结果**：四张原始表按业务键合并并回查原划分表后，得到 84,776 条记录：Train 67,820、Validation 8,478、Test 8,478；重复 `load_id` 为 0。对照旧 Train 的 800 条 Task A 样本，所有已展示字段与源记录一致，字段不一致次数为 0。

**结论**：业务表连接和 Task A 字段映射通过核对，可以作为新增指令样本的数据来源。下一步检查 Task B 的时间标签，再确定扩展抽样。

### 4.1.3：Task B 标签复核

这一步检查精确时间比较，并将旧 Train/Val 的标准答案与源记录重新计算的标签对照。Test 只汇总总体分布，不展示逐条答案。

In [ ]:
import json
import pandas as pd
from collections import Counter

# 解析完整时间戳，保留秒以下精度
scheduled = pd.to_datetime(
    business["scheduled_datetime"], errors="coerce"
)
actual = pd.to_datetime(
    business["actual_datetime"], errors="coerce"
)

invalid_time = scheduled.isna() | actual.isna()
business = business.copy()
business["recomputed_label"] = None
business.loc[~invalid_time, "recomputed_label"] = (
    (actual[~invalid_time] > scheduled[~invalid_time])
    .map({True: "delayed", False: "on_time"})
)

print("无法解析时间的业务记录：", int(invalid_time.sum()))
print("\n各组标签分布：")
print(pd.crosstab(business["split"], business["recomputed_label"]))

# 旧指令中已有的 Task B 标签，只核对 Train 和 Val
labels_by_id = business.set_index("load_id")["recomputed_label"]
mismatch_ids = []
checked = Counter()

for split in ("train", "val"):
    for sample in samples[split]:
        if sample["task"] != "B":
            continue

        checked[split] += 1
        old_label = json.loads(
            sample["messages"][-1]["content"]
        )["delay_label"]
        new_label = labels_by_id.loc[sample["source_load_id"]]

        if old_label != new_label:
            mismatch_ids.append(sample["sample_id"])

print("\n旧 Task B 标签核对数：", dict(checked))
print("标签不一致数：", len(mismatch_ids))
print("不一致样本 ID（最多 10 个）：", mismatch_ids[:10])

84,776 条记录的计划和实际时间都可解析；旧 Train 800 条、Val 100 条 Task B 标签与精确时间比较结果全部一致。这里核对的是数据标签，不是模型预测准确率。

### 4.1.3 Task B 时间标签复核

**方法**：将计划交付时间与实际完成时间解析为完整时间戳。实际时间晚于计划时间标为 `delayed`，相等或更早标为 `on_time`。再将重新计算的结果与旧 Train、Val 中的 Task B 标准答案逐条对照。

| 检查项 | 实测结果 |
| --- | ---: |
| 无法解析时间的业务记录 | 0 / 84,776 |
| 旧 Train 标签不一致 | 0 / 800 |
| 旧 Val 标签不一致 | 0 / 100 |

| 原始划分 | delayed | on_time |
| --- | ---: | ---: |
| Train | 45,492 | 22,328 |
| Validation | 5,687 | 2,791 |
| Test | 5,687 | 2,791 |

**结论**：现有 Task B 标签与业务规则一致，可以用相同的精确时间比较方法为新增样本生成标签。此处的 Test 数字仅用于数据分布检查，尚未运行模型测试。

下一步是只确定新增运单 ID：Val 从 100 扩至 500 条业务记录（A/B 共 1,000 条指令），Test 从 100 扩至 1,000 条业务记录（A/B 共 2,000 条指令）。保留旧 ID，并按原有 delayed / on_time 比例抽取新增记录。运行下面代码：

In [ ]:
import random
from collections import Counter

SEED = 42

# 每条业务记录生成 A、B 各一条指令
target_records = {"validation": 500, "test": 1000}
old_file_name = {"validation": "val", "test": "test"}

selected_ids = {}

for split, target in target_records.items():
    # 保留正式版 JSONL 已使用的全部业务记录
    old_ids = {
        x["source_load_id"]
        for x in samples[old_file_name[split]]
    }

    pool = business.loc[
        business["split"] == split,
        ["load_id", "recomputed_label"]
    ].copy()

    # 按该划分的原有类别比例确定目标数量
    delayed_rate = (pool["recomputed_label"] == "delayed").mean()
    target_delayed = round(target * delayed_rate)
    target_on_time = target - target_delayed

    old_labels = pool.loc[
        pool["load_id"].isin(old_ids), "recomputed_label"
    ].value_counts()

    needed = {
        "delayed": target_delayed - old_labels.get("delayed", 0),
        "on_time": target_on_time - old_labels.get("on_time", 0),
    }

    # 如果旧样本某类已超过目标比例，暂停检查，不强行丢弃旧样本
    assert all(n >= 0 for n in needed.values()), (
        f"{split}：旧样本类别数超过目标，请检查：{needed}"
    )

    rng = random.Random(SEED)
    new_ids = set()

    for label in ("delayed", "on_time"):
        candidates = sorted(
            pool.loc[
                (pool["recomputed_label"] == label)
                & (~pool["load_id"].isin(old_ids)),
                "load_id"
            ].tolist()
        )
        assert len(candidates) >= needed[label]
        new_ids.update(rng.sample(candidates, needed[label]))

    selected_ids[split] = old_ids | new_ids

    selected_labels = pool.loc[
        pool["load_id"].isin(selected_ids[split]),
        "recomputed_label"
    ]

    print(f"\n{split}")
    print("保留旧业务记录：", len(old_ids))
    print("新增业务记录：", len(new_ids))
    print("最终业务记录：", len(selected_ids[split]))
    print("预计指令样本：", 2 * len(selected_ids[split]))
    print("标签分布：", selected_labels.value_counts().to_dict())
    print("旧记录全部保留：", old_ids <= selected_ids[split])

print("\nVal / Test 记录交集：",
      len(selected_ids["validation"] & selected_ids["test"]))

### 4.1.4 扩展样本的业务记录选择

在原有 Validation 和 Test 业务记录池中，固定随机种子 `42`，按延误标签比例抽取新增记录。每条业务记录生成 Task A、Task B 各一条指令。

| 数据集 | 保留旧记录 | 新增记录 | 最终记录 | 最终指令数 |
| --- | ---: | ---: | ---: | ---: |
| Validation | 100 | 400 | 500 | 1,000 |
| Test | 100 | 900 | 1,000 | 2,000 |

Validation 的 500 条记录中，`delayed` 335 条、`on_time` 165 条；Test 的 1,000 条记录中分别为 671 条、329 条。旧记录全部保留，两个集合的记录交集为 0。

**方法与作用**：按业务记录而非单条指令抽样，保证同一运单的 A/B 样本在同一集合；固定随机种子和类别比例，使扩展过程可复现，并避免评测集的延误类别明显失衡。

> 当前只完成记录选择，尚未写出新增 JSONL 或进行模型评测。

接下来生成指令前，我要先核对旧样本的具体文本格式，尤其是 Task A 各字段的中文表达、数字格式，以及 Task B 时间表述。虽然 dataset_info.json 给了模板，但还需要确认实际旧样本如何使用它们，避免新样本的格式与 Phase 1 不一致。运行下面的只读检查格；它只展示旧 Train/Val 的少量样本，不查看 Test 内容：

In [ ]:
import json

for split in ("train", "val"):
    print(f"\n===== {split}：Task A 示例 =====")
    task_a = next(x for x in samples[split] if x["task"] == "A")
    print("template_id：", task_a["template_id"])
    print("omitted_fields：", task_a["omitted_fields"])
    print("用户输入：", task_a["messages"][1]["content"])
    print("标准答案：", task_a["messages"][2]["content"])

    print(f"\n===== {split}：Task B 示例 =====")
    task_b = next(x for x in samples[split] if x["task"] == "B")
    print("template_id：", task_b["template_id"])
    print("用户输入：", task_b["messages"][1]["content"])
    print("标准答案：", task_b["messages"][2]["content"])

格式已确认。还有一个需要正确处理的细节：原始业务表中部分记录的 truck_id 本来就是缺失值；这类字段应在 Task A 答案中写 null，不能生成字符串 "nan"。下面先在内存中生成新增样本，保留旧 JSONL 原样，暂不保存文件。

In [ ]:
import json
import random
import pandas as pd
from collections import Counter

# 使用 Phase 1 数据说明中的原版指令与模板
system_prompt = info["system_prompt"]
instruction_a = info["task_A_instruction"]
instruction_b = info["task_B_instruction"]
answer_fields = info["task_A_fields"]

# 原始业务表字段 -> Task A 答案字段
source_columns = {
    "load_id": "load_id",
    "truck_id": "truck_id",
    "origin_city": "origin_city",
    "destination_city": "destination_city",
    "revenue_usd": "revenue",
    "fuel_used_gallons": "fuel_gallons_used",
    "scheduled_delivery": "scheduled_datetime",
    "actual_delivery": "actual_datetime",
}

# Task A 用户输入中使用的中文字段名称
chinese_names = {
    "load_id": "运单编号",
    "truck_id": "车辆编号",
    "origin_city": "出发城市",
    "destination_city": "目的城市",
    "revenue_usd": "运单收入（美元）",
    "fuel_used_gallons": "实际耗油量（加仑）",
    "scheduled_delivery": "计划交付时间",
    "actual_delivery": "实际交付时间",
}

numeric_fields = {"revenue_usd", "fuel_used_gallons"}
business_index = business.set_index("load_id", drop=False)

# 分别生成两组，随机数生成器独立，确保结果可复现
expanded_samples = {}
generation_stats = {}

for split, old_key, seed in [
    ("validation", "val", 1042),
    ("test", "test", 2042),
]:
    rng = random.Random(seed)
    old_items = samples[old_key]
    old_ids = {x["source_load_id"] for x in old_items}
    new_ids = sorted(selected_ids[split] - old_ids)

    # 对约 25% 的新增 Task A 记录主动隐藏 1～2 个字段
    # 源数据本身缺失的字段也会填 null，但不计入“主动隐藏”比例
    omit_ids = set(rng.sample(new_ids, round(len(new_ids) * 0.25)))
    new_items = []
    inherent_missing = Counter()
    planned_omission = Counter()

    for load_id in new_ids:
        row = business_index.loc[load_id]

        # 生成 Task A 标准答案，保留完整时间格式
        answer_a = {}
        for field in answer_fields:
            value = row[source_columns[field]]
            if pd.isna(value):
                answer_a[field] = None
                inherent_missing[field] += 1
            elif field in numeric_fields:
                answer_a[field] = float(value)
            else:
                answer_a[field] = str(value)

        # 始终保留 load_id；只从其他有值的字段中主动隐藏
        hide_candidates = [
            field for field in answer_fields
            if field != "load_id" and answer_a[field] is not None
        ]
        hidden = set()
        if load_id in omit_ids:
            hide_count = min(rng.choice([1, 2]), len(hide_candidates))
            hidden = set(rng.sample(hide_candidates, hide_count))
            planned_omission.update(hidden)

        # 输入未提供的字段，答案必须为 JSON null
        for field in hidden:
            answer_a[field] = None

        shown_fields = [
            field for field in answer_fields
            if answer_a[field] is not None
        ]
        rng.shuffle(shown_fields)

        details = "；".join(
            f"{chinese_names[field]}：{answer_a[field]}"
            for field in shown_fields
        ) + "。"

        templates_a = info["task_A_templates"][split]
        template_a_index = rng.randrange(len(templates_a))
        user_a = instruction_a + templates_a[template_a_index].format(
            details=details
        )

        omitted_fields = [
            field for field in answer_fields
            if field not in shown_fields
        ]

        sample_a = {
            "sample_id": f"{load_id}_A",
            "source_load_id": load_id,
            "source_trip_id": str(row["trip_id"]),
            "split": split,
            "task": "A",
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_a},
                {
                    "role": "assistant",
                    "content": json.dumps(answer_a, ensure_ascii=False),
                },
            ],
            "template_id": f"{split}_A_{template_a_index + 1}",
            "omitted_fields": omitted_fields,
            "shown_fields": shown_fields,
        }

        # Task B 只从源记录的两个时间戳生成题目和标签
        scheduled_text = str(row["scheduled_datetime"])
        actual_text = str(row["actual_datetime"])
        templates_b = info["task_B_templates"][split]
        template_b_index = rng.randrange(len(templates_b))
        user_b = instruction_b + templates_b[template_b_index].format(
            scheduled=scheduled_text,
            actual=actual_text,
        )
        answer_b = {"delay_label": row["recomputed_label"]}

        sample_b = {
            "sample_id": f"{load_id}_B",
            "source_load_id": load_id,
            "source_trip_id": str(row["trip_id"]),
            "split": split,
            "task": "B",
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_b},
                {
                    "role": "assistant",
                    "content": json.dumps(answer_b, ensure_ascii=False),
                },
            ],
            "template_id": f"{split}_B_{template_b_index + 1}",
            "omitted_fields": [],
            "shown_fields": [
                "scheduled_delivery", "actual_delivery"
            ],
        }

        new_items.extend([sample_a, sample_b])

    # 旧样本保持原对象与原顺序；新增样本追加在后面
    expanded_samples[split] = old_items + new_items
    generation_stats[split] = {
        "old_instructions": len(old_items),
        "new_instructions": len(new_items),
        "total_instructions": len(expanded_samples[split]),
        "actively_omitted_A_records": len(omit_ids),
        "source_missing_fields": dict(inherent_missing),
        "actively_omitted_fields": dict(planned_omission),
    }

for split, stats in generation_stats.items():
    print(f"\n===== {split} =====")
    for key, value in stats.items():
        print(f"{key}: {value}")

# 只查看一条新增 Validation 示例；Test 仍不展示题目或答案
first_new_val = expanded_samples["validation"][len(samples["val"])]
print("\n新增 Validation Task A 示例：")
print(json.dumps(first_new_val, ensure_ascii=False, indent=2)[:2500])

生成数量正确：Validation 新增 800 条、合计 1,000 条；Test 新增 1,800 条、合计 2,000 条。主动隐藏字段的新增 Task A 比例都是 25%；源表中缺失的 truck_id 也被单独识别。示例里的输入和答案格式与旧 Val 一致。

### 4.1.5 新增指令样本生成

沿用 Phase 1 的系统指令、任务定义和 Validation/Test 文本模板，将选定的业务记录转换为 Task A 信息提取与 Task B 延误判断样本。旧样本保持原样，新增样本追加在后。

| 数据集 | 旧指令 | 新指令 | 合计 |
| --- | ---: | ---: | ---: |
| Validation | 200 | 800 | 1,000 |
| Test | 200 | 1,800 | 2,000 |

新增 Task A 中，Validation 有 100/400 条、Test 有 225/900 条记录主动隐藏字段，均为 **25%**。源数据本身另有 6 条和 14 条记录缺少 `truck_id`，对应答案应填 `null`。

**技术与作用：**使用固定随机种子、原版模板和 JSONL 对话结构生成可复现的评测样本；对缺失字段输出 JSON `null`，检验模型是否会编造信息。以上是生成统计，最终质量结论以接下来的全量检查为准。

In [ ]:
import json
from collections import Counter, defaultdict

issues = Counter()
train_ids = {x["source_load_id"] for x in samples["train"]}
all_ids = {}

for split, old_key, expected_total in [
    ("validation", "val", 1000),
    ("test", "test", 2000),
]:
    items = expanded_samples[split]
    old_items = samples[old_key]
    new_items = items[len(old_items):]
    tasks_by_load = defaultdict(list)
    sample_ids = []

    if len(items) != expected_total:
        issues[f"{split}:总数量"] += 1

    # 旧样本应原样保留，顺序也不改变
    if items[:len(old_items)] != old_items:
        issues[f"{split}:旧样本改变"] += 1

    for x in items:
        load_id = x["source_load_id"]
        sample_ids.append(x["sample_id"])
        tasks_by_load[load_id].append(x["task"])

        if x["split"] != split:
            issues[f"{split}:split字段"] += 1
        if load_id not in selected_ids[split]:
            issues[f"{split}:不在选定记录中"] += 1
        if load_id in train_ids:
            issues[f"{split}:与训练样本重叠"] += 1
        if assignments[load_id]["split"] != split:
            issues[f"{split}:与原划分表不符"] += 1
        if x["sample_id"] != f"{load_id}_{x['task']}":
            issues[f"{split}:sample_id格式"] += 1

    if len(sample_ids) != len(set(sample_ids)):
        issues[f"{split}:重复sample_id"] += (
            len(sample_ids) - len(set(sample_ids))
        )

    pair_errors = sum(
        sorted(tasks) != ["A", "B"]
        for tasks in tasks_by_load.values()
    )
    issues[f"{split}:A/B配对"] += pair_errors
    all_ids[split] = set(tasks_by_load)

    # 逐条复核新增样本；旧样本的划分和标签已在前面核对
    for x in new_items:
        row = business_index.loc[x["source_load_id"]]

        try:
            answer = json.loads(x["messages"][-1]["content"])
        except (ValueError, TypeError):
            issues[f"{split}:答案JSON无效"] += 1
            continue

        if x["task"] == "A":
            if set(answer) != set(answer_fields):
                issues[f"{split}:A字段结构"] += 1
                continue

            if set(x["shown_fields"]) & set(x["omitted_fields"]):
                issues[f"{split}:展示与缺失字段重叠"] += 1
            if set(x["shown_fields"]) | set(x["omitted_fields"]) != set(answer_fields):
                issues[f"{split}:字段覆盖不完整"] += 1

            for field in answer_fields:
                value = row[source_columns[field]]
                expected = (
                    None if pd.isna(value)
                    else float(value) if field in numeric_fields
                    else str(value)
                )

                if field in x["omitted_fields"]:
                    if answer[field] is not None:
                        issues[f"{split}:A缺失值标签"] += 1
                elif answer[field] != expected:
                    issues[f"{split}:A源字段不符"] += 1

            # 输入中应恰好出现所有展示字段及其值
            user_text = x["messages"][1]["content"]
            for field in x["shown_fields"]:
                fragment = f"{chinese_names[field]}：{answer[field]}"
                if fragment not in user_text:
                    issues[f"{split}:A输入字段不符"] += 1

        elif x["task"] == "B":
            if answer != {"delay_label": row["recomputed_label"]}:
                issues[f"{split}:B时间标签"] += 1

            template_number = int(x["template_id"].rsplit("_", 1)[-1])
            template = info["task_B_templates"][split][template_number - 1]
            expected_user = instruction_b + template.format(
                scheduled=str(row["scheduled_datetime"]),
                actual=str(row["actual_datetime"]),
            )
            if x["messages"][1]["content"] != expected_user:
                issues[f"{split}:B输入时间不符"] += 1
        else:
            issues[f"{split}:未知任务"] += 1

    print(f"{split}: {len(items)} 条指令，"
          f"{len(tasks_by_load)} 条业务记录，"
          f"新增 {len(new_items)} 条")

issues["validation/test记录重叠"] = len(
    all_ids["validation"] & all_ids["test"]
)

print("\n===== 质量检查异常 =====")
nonzero = {name: count for name, count in issues.items() if count}
print(nonzero if nonzero else "全部为 0")

In [ ]:
import json
import hashlib
import zipfile
from pathlib import Path

old_zip_path = Path("/content/FleetMind_LLM_Phase1_Dataset.zip")
new_zip_path = Path("/content/FleetMind_LLM_Phase4_Evaluation_Dataset.zip")
old_prefix = "fleetmind_phase1_v1/"
new_prefix = "fleetmind_phase4_evaluation/"

def sha256_file(path):
    """计算文件指纹，便于以后确认数据版本。"""
    return hashlib.sha256(path.read_bytes()).hexdigest()

def encode_new_samples(items):
    """把新增样本编码为 JSONL，每行一个 JSON 对象。"""
    return "".join(
        json.dumps(item, ensure_ascii=False) + "\n"
        for item in items
    ).encode("utf-8")

with zipfile.ZipFile(old_zip_path) as old_zip:
    # 训练集直接复制原文件字节，不重新生成
    train_bytes = old_zip.read(old_prefix + "train.jsonl")

    # 原有 Val/Test 字节保持不变，仅在末尾追加新样本
    expanded_bytes = {}
    for split, old_name in [("validation", "val"), ("test", "test")]:
        old_bytes = old_zip.read(old_prefix + f"{old_name}.jsonl")
        new_items = expanded_samples[split][len(samples[old_name]):]

        expanded_bytes[old_name] = (
            old_bytes.rstrip(b"\r\n")
            + b"\n"
            + encode_new_samples(new_items)
        )

    # 原始划分表和 Phase 1 说明原样保存，方便追溯
    assignments_bytes = old_zip.read(
        old_prefix + "split_assignments.csv"
    )
    original_info_bytes = old_zip.read(
        old_prefix + "dataset_info.json"
    )

# 单独记录 Phase 4 的实际生成规则与数量
phase4_info = {
    "project": "FleetMind-LLM",
    "stage": "Phase 4 / Step 4.1",
    "source_type": "synthetic",
    "source_dataset_zip": "Fleetmind_dataset.zip",
    "phase1_dataset_sha256": sha256_file(old_zip_path),
    "raw_dataset_sha256": sha256_file(
        Path("/content/Fleetmind_dataset.zip")
    ),
    "business_record_selection_seed": 42,
    "instruction_generation_seeds": {
        "validation": 1042,
        "test": 2042,
    },
    "instruction_counts": {
        "train": 1600,
        "validation": 1000,
        "test": 2000,
    },
    "business_record_counts": {
        "train_used_for_training": 800,
        "validation": 500,
        "test": 1000,
    },
    "old_samples_preserved": True,
    "new_task_A_active_omission_rate": 0.25,
    "test_policy": "封存；最终模型和评测方案确定后使用",
}

# 创建新压缩包，不覆盖 Phase 1 的正式数据包
with zipfile.ZipFile(
    new_zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as new_zip:
    new_zip.writestr(new_prefix + "train.jsonl", train_bytes)
    new_zip.writestr(
        new_prefix + "val.jsonl", expanded_bytes["val"]
    )
    new_zip.writestr(
        new_prefix + "test.jsonl", expanded_bytes["test"]
    )
    new_zip.writestr(
        new_prefix + "split_assignments.csv",
        assignments_bytes
    )
    new_zip.writestr(
        new_prefix + "phase1_dataset_info.json",
        original_info_bytes
    )
    new_zip.writestr(
        new_prefix + "phase4_dataset_info.json",
        json.dumps(phase4_info, ensure_ascii=False, indent=2)
    )

# 从刚写好的 ZIP 重新读取，核对数量和旧文件的完整保留
with zipfile.ZipFile(new_zip_path) as check_zip:
    expected = {"train": 1600, "val": 1000, "test": 2000}

    for split, count in expected.items():
        data = check_zip.read(new_prefix + f"{split}.jsonl")
        lines = data.decode("utf-8").splitlines()
        assert len(lines) == count, (split, len(lines))
        for line in lines:
            json.loads(line)
        print(f"{split}: {len(lines)} 条，JSONL 回读通过")

    assert (
        check_zip.read(new_prefix + "train.jsonl")
        == train_bytes
    )
    assert (
        check_zip.read(new_prefix + "val.jsonl")
        .startswith(
            old_zip_path.exists() and
            zipfile.ZipFile(old_zip_path).read(
                old_prefix + "val.jsonl"
            ).rstrip(b"\r\n") + b"\n"
        )
    )

print("\n新数据包：", new_zip_path)
print("大小：", round(new_zip_path.stat().st_size / 1024**2, 2), "MB")
print("SHA-256：", sha256_file(new_zip_path))

In [ ]:
from pathlib import Path
import shutil
import hashlib

source = Path("/content/FleetMind_LLM_Phase4_Evaluation_Dataset.zip")
destination_dir = Path("/content/drive/MyDrive/FleetMind-LLM")
destination_dir.mkdir(parents=True, exist_ok=True)
destination = destination_dir / source.name

def file_hash(path):
    # 核对复制前后的文件内容完全一致
    return hashlib.sha256(path.read_bytes()).hexdigest()

# 如果目标位置已有不同内容的同名文件，停止而不覆盖
if destination.exists() and file_hash(destination) != file_hash(source):
    raise FileExistsError(f"Drive 中已有不同内容的同名文件：{destination}")

shutil.copy2(source, destination)
assert file_hash(source) == file_hash(destination)

print("已保存：", destination)
print("SHA-256：", file_hash(destination))

In [ ]:
from google.colab import files

files.download("/content/FleetMind_LLM_Phase4_Evaluation_Dataset.zip")

## Step 4.1 阶段结论：评测数据扩展与质量检查

本步骤沿用 Phase 1 的原始业务记录划分和指令格式，保留所有旧样本，将 Validation 扩展至 **1,000 条指令**、Test 扩展至 **2,000 条指令**。Train 保持 1,600 条。

| 检查内容 | 实测结果 |
| --- | --- |
| 合并并回查原划分的业务记录 | 84,776 条 |
| Train / Validation / Test 原业务池 | 67,820 / 8,478 / 8,478 条 |
| 扩展后指令数量 | 1,600 / 1,000 / 2,000 条 |
| 旧 Train Task A 与源字段不一致 | 0 / 800 |
| 旧 Train、Val Task B 标签不一致 | 0 / 900 |
| 无法解析交付时间 | 0 |
| A/B 配对、重复 ID、跨集合记录重叠 | 0 项异常 |
| 新增样本全量质量检查 | 0 项异常 |
| 扩展版 JSONL 回读 | 三组全部通过 |


新增 Task A 样本中，Validation 的 100/400 条、Test 的 225/900 条业务记录主动隐藏字段，均为 25%。源数据本身缺少 `truck_id` 的记录另按 `null` 处理。原数据来源标注为**合成物流数据**，后续报告和简历也应如实说明。

**数据版本：**`FleetMind_LLM_Phase4_Evaluation_Dataset.zip`，SHA-256：
`b419b995d4ec7505cea36ac415c60070b3e16f449558a669ce3735d53d1e582c`

**评测边界**：旧 Val 200 已参与 Phase 3 模型选择，因此扩展后的 Val 仍用于开发和方法比较。扩展后的 Test 保持封存，待最终模型与评测方案确定后再使用。

## 4.2 四种方法对比

在同一份 Val 1,000 上比较：

| 方法 | 做法 | 比较目的 |
| --- | --- | --- |
| Rule-based | 使用确定性规则完成适用任务，尤其是 Task B 时间判断 | 提供业务规则参考 |
| Base（Zero-shot） | 原始 Qwen 直接按照任务指令回答 | 测量未经微调的能力 |
| Base（Few-shot） | 在 Prompt 中提供少量训练集示例 | 测量示例提示带来的改善 |
| Qwen + LoRA | 使用 Phase 3 已训练的模型回答 | 测量领域微调的效果 |

Few-shot 示例**只从训练集选取**，不从 Val 或 Test 取例子。各方法使用一致的任务定义、输出要求和评分规则；记录实际 Prompt 与推理配置。

作用：弄清性能来自规则、提示示例，还是 LoRA 微调。

### 4.2.1 Task B 规则基线

从用户输入中识别计划交付时间与实际完成时间，使用完整时间戳进行比较：实际晚于计划为 `delayed`，否则为 `on_time`。记录无法解析的数量，并计算分类指标。

**作用**：提供一个明确的业务规则参照，用来判断语言模型在时间比较任务上的表现。此处仅使用 Validation，不使用封存的 Test。

In [ ]:
import re
from datetime import datetime
from collections import Counter

# 支持带微秒的时间戳；保留原始精度
time_pattern = r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}(?:\.\d+)?)"

def rule_predict_b(user_text):
    """从 Validation 题目中提取两种模板的时间并判断延误。"""
    plan_patterns = [
        rf"原计划{time_pattern}",
        rf"约定计划为{time_pattern}",
    ]
    actual_patterns = [
        rf"实际完成于{time_pattern}",
        rf"实际为{time_pattern}",
    ]

    plan_match = next(
        (m for p in plan_patterns if (m := re.search(p, user_text))),
        None
    )
    actual_match = next(
        (m for p in actual_patterns if (m := re.search(p, user_text))),
        None
    )

    if plan_match is None or actual_match is None:
        return None

    plan = datetime.fromisoformat(plan_match.group(1))
    actual = datetime.fromisoformat(actual_match.group(1))
    return "delayed" if actual > plan else "on_time"

# 只使用扩展后的 Validation Task B
val_b = [
    item for item in expanded_samples["validation"]
    if item["task"] == "B"
]
assert len(val_b) == 500

results = []

for item in val_b:
    truth = json.loads(
        item["messages"][-1]["content"]
    )["delay_label"]
    prediction = rule_predict_b(item["messages"][1]["content"])

    results.append({
        "sample_id": item["sample_id"],
        "truth": truth,
        "prediction": prediction,
    })

unparsed = [r for r in results if r["prediction"] is None]
correct = sum(r["prediction"] == r["truth"] for r in results)

# delayed 为重点关注的正类；无法解析也视为未正确预测
tp = sum(r["truth"] == "delayed" and r["prediction"] == "delayed"
         for r in results)
fp = sum(r["truth"] != "delayed" and r["prediction"] == "delayed"
         for r in results)
fn = sum(r["truth"] == "delayed" and r["prediction"] != "delayed"
         for r in results)

accuracy = correct / len(results)
precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

print("Validation Task B 样本数：", len(results))
print("无法解析：", len(unparsed))
print("正确数：", correct)
print(f"Accuracy：{accuracy:.4%}")
print(f"delayed Precision：{precision:.4%}")
print(f"delayed Recall：{recall:.4%}")
print(f"delayed F1：{f1:.4%}")
print("混淆计数 TP/FP/FN：", tp, fp, fn)
print("错误样本 ID（最多 10 个）：",
      [r["sample_id"] for r in results
       if r["prediction"] != r["truth"]][:10])

**实测结果**：扩展后的 Validation 集包含 500 条 Task B 样本。规则方法成功解析全部 500 条时间记录，正确判断 500 条；Accuracy 为 100%，以 `delayed` 为正类的 Precision、Recall、F1 均为 100%。TP=335、FP=0、FN=0，没有错误样本。

**结论**：在当前数据和题目模板下，精确时间戳比较可以完整解决 Task B。后续评测 LoRA 时，应如实与这个规则基线比较，不能只与原始 Qwen 比较。这项结果仅属于 Validation，不代表独立 Test 成绩。

### 4.2.2 原始 Qwen：Zero-shot 基线

**目的**：了解未经微调的 Qwen3-0.6B 在 Task A 信息提取和 Task B 延误判断上的表现，为后续判断 LoRA 是否带来提升提供参照。

**做法**：直接向原始模型提供与当前数据一致的任务指令，不给示例。先运行少量 Validation 样本，确认输出格式和速度，再评测完整的 1,000 条 Validation 指令。

**记录内容**：Task A 的 JSON 与字段指标；Task B 的 Accuracy、Precision、Recall、F1；同时保存逐条预测，便于后续分析错误。

> Test 集继续封存。具体结果在运行后填写。

| 方法 | 模型会看到什么 |
| --- | --- |
| **Zero-shot** | 任务说明 + 当前题目；没有示范答案 |
| **Few-shot** | 任务说明 + 几条训练集中的示范题和答案 + 当前题目 |
| **LoRA** | 模型参数已经通过训练样本更新，再回答当前题目 |

测 Zero-shot，是想知道原始 Qwen 在没经过 FleetMind-LLM 微调时能做到什么程度。之后与 Few-shot 和 LoRA 在同一批 Val 样本上比较，才能判断示例提示和微调各自带来了多少变化。

In [ ]:
import sys
import torch
from importlib.metadata import version, PackageNotFoundError

# 检查当前 Colab 是否有可用 GPU
print("Python：", sys.version.split()[0])
print("PyTorch：", torch.__version__)
print("CUDA 可用：", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU：", torch.cuda.get_device_name(0))

# 读取实际安装版本；不假定新 Notebook 继承 Phase 3 环境
for package in ("transformers", "accelerate", "peft", "trl"):
    try:
        print(f"{package}：{version(package)}")
    except PackageNotFoundError:
        print(f"{package}：未安装")

###Zero-shot 推理试跑

**目的**：确认原始 Qwen3-0.6B 能在当前 T4 环境中正常加载，并检查它对 Task A、Task B 的输出格式和推理速度。

**做法**：从 Validation 集各取 2 条 Task A 和 Task B 样本，只向模型提供当前题目的系统指令与用户输入，不提供示例或标准答案。关闭思考模式，记录模型回答和每条推理耗时。

**检查重点**：模型是否正常生成；回答是否符合要求的 JSON 格式；是否出现多余解释、空回答或明显截断。

> 这 4 条仅用于检查运行流程，不作为正式评测成绩。完整结果将在同一套设置下对全部 1,000 条 Validation 样本运行后统计。

In [ ]:
import json
import hashlib
import zipfile
from pathlib import Path
from google.colab import files

# 从桌面选择 FleetMind_LLM_Phase4_Evaluation_Dataset.zip
uploaded = files.upload()
expected_name = "FleetMind_LLM_Phase4_Evaluation_Dataset.zip"

if expected_name not in uploaded:
    raise FileNotFoundError(f"请选择 {expected_name}")

dataset_path = Path("/content") / expected_name

# 与 4.1 记录的文件指纹比较，确认上传的是同一版本
actual_hash = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
expected_hash = "b419b995d4ec7505cea36ac415c60070b3e16f449558a669ce3735d53d1e582c"
assert actual_hash == expected_hash, (
    f"文件版本不一致：{actual_hash}"
)

# 只恢复 Validation 样本，不读取 Test
with zipfile.ZipFile(dataset_path) as archive:
    val_lines = archive.read(
        "fleetmind_phase4_evaluation/val.jsonl"
    ).decode("utf-8").splitlines()

val_samples = [json.loads(line) for line in val_lines]
assert len(val_samples) == 1000

print("数据包版本核对通过")
print("Validation：", len(val_samples))
print("Task A：", sum(x["task"] == "A" for x in val_samples))
print("Task B：", sum(x["task"] == "B" for x in val_samples))

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-0.6B"
assert torch.cuda.is_available(), "请先切换到 Colab GPU 运行时"

# 原始 Qwen 模型，不加载 Phase 3 的 LoRA adapter
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="cuda:0",
)
base_model.eval()

def zero_shot_predict(sample, max_new_tokens=384):
    """输入一条题目，返回原始模型的回答和推理耗时。"""

    # 只给模型 system 和 user；第三条 assistant 是标准答案
    prompt_messages = sample["messages"][:2]

    # 新版 Transformers 返回包含 input_ids、attention_mask 的对象
    model_inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    )

    # 将输入中的各个张量移到模型所在的 GPU
    model_inputs = {
        name: tensor.to(base_model.device)
        for name, tensor in model_inputs.items()
    }
    input_length = model_inputs["input_ids"].shape[-1]

    start = time.perf_counter()
    with torch.inference_mode():
        # ** 将 input_ids 和 attention_mask 分别传给 generate
        output_ids = base_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - start

    # 输出含原输入，只解码新生成的回答
    generated_ids = output_ids[0, input_length:]
    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return answer, elapsed

print("原始模型已加载：", MODEL_ID)
print("运行设备：", base_model.device)

In [ ]:
# 两条 Task A + 两条 Task B；仅检查流程，不计算正式成绩
pilot = (
    [x for x in val_samples if x["task"] == "A"][:2]
    + [x for x in val_samples if x["task"] == "B"][:2]
)

for sample in pilot:
    prediction, seconds = zero_shot_predict(sample)
    print(
        f"\n{sample['sample_id']} | "
        f"Task {sample['task']} | {seconds:.2f} 秒"
    )
    print("模型回答：", prediction)

### 正式对比前核对基座模型

**目的：**确认 Zero-shot 使用的原始 Qwen，与 Phase 3 训练 LoRA 时使用的基座模型一致。

**通过条件：**配置记录的基座模型与当前 `MODEL_ID` 一致。确认后再运行 Validation 1,000 条的 Zero-shot 生成。

In [ ]:
import json
import os
import zipfile
from google.colab import files

phase3_zip = "/content/FleetMind_LLM_Phase3_Results.zip"

# Colab 换了运行时之后，原来上传的文件可能不在。
# 只有找不到备份包时才要求上传。
if not os.path.exists(phase3_zip):
    print("请上传 FleetMind_LLM_Phase3_Results.zip")
    uploaded = files.upload()
    assert os.path.basename(phase3_zip) in uploaded, "没有上传指定的 Phase 3 备份包"

with zipfile.ZipFile(phase3_zip, "r") as archive:
    # 优先读取最终选出的 best_adapter 配置，
    # 避免误读训练过程中其他 checkpoint 的配置。
    candidates = [
        name for name in archive.namelist()
        if name.endswith("adapter_config.json")
    ]
    assert candidates, "备份包中找不到 adapter_config.json"

    best_candidates = [
        name for name in candidates if "best_adapter" in name
    ]
    config_path = best_candidates[0] if best_candidates else candidates[0]

    adapter_config = json.loads(
        archive.read(config_path).decode("utf-8")
    )

phase3_base = adapter_config.get("base_model_name_or_path")

print("读取的配置文件：", config_path)
print("Phase 3 基座模型：", phase3_base)
print("当前 Zero-shot 模型：", MODEL_ID)
print("LoRA r：", adapter_config.get("r"))
print("LoRA alpha：", adapter_config.get("lora_alpha"))
print("LoRA dropout：", adapter_config.get("lora_dropout"))
print("目标模块：", adapter_config.get("target_modules"))

# 两个模型名称必须一致，才能直接用当前基座做微调前后对比。
assert phase3_base == MODEL_ID, (
    "基座模型名称不一致；先核对配置，暂不要运行全量 Zero-shot。"
)
print("基座模型核对通过，可以继续全量 Zero-shot 评测。")

### Zero-shot 全量验证

目的：评估未经 FleetMind 微调的原始 Qwen，在物流信息提取（Task A）和延误判断（Task B）上的表现，作为后续模型对比的基线。

评测数据：扩展后的 Validation，共 1,000 条指令，Task A 和 Task B 各 500 条。Test 暂不使用。

统一设置：不提供示范题；关闭 thinking；使用确定性生成；最多生成 384 个新 token。模型只看到任务说明和当前题目，看不到标准答案。

结果保存：逐条记录模型原始回答和耗时。后续单独计算严格 JSON 有效率、字段准确率及分类指标，不在生成过程中修改模型回答。

In [ ]:
import json
import os
import shutil
from collections import Counter

# 只对已经加载的扩展版 Validation 进行评测。
# 先核对数量，防止误用旧版 200 条验证集。
task_counts = Counter(sample["task"] for sample in val_samples)
assert len(val_samples) == 1000, f"Validation 数量不对：{len(val_samples)}"
assert task_counts == {"A": 500, "B": 500}, f"Task 数量不对：{task_counts}"

# 优先保存到已挂载的 Google Drive；如果尚未挂载，则保存在当前 Colab。
# 保存在 /content 的文件需要在运行时结束前下载，否则可能丢失。
drive_root = "/content/drive/MyDrive"
save_dir = (
    os.path.join(drive_root, "FleetMind-LLM", "phase4")
    if os.path.isdir(drive_root)
    else "/content"
)
os.makedirs(save_dir, exist_ok=True)
result_path = os.path.join(
    save_dir, "phase4_zero_shot_val_predictions.jsonl"
)

# 如果之前运行中断，读取已经保存的样本编号。
# 重新运行这个单元格时，会跳过它们，避免重复生成。
completed_ids = set()
if os.path.exists(result_path):
    with open(result_path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                record = json.loads(line)
                sample_id = record["sample_id"]
                assert sample_id not in completed_ids, f"结果文件有重复：{sample_id}"
                completed_ids.add(sample_id)

valid_ids = {sample["sample_id"] for sample in val_samples}
assert completed_ids <= valid_ids, "结果文件中有不属于当前 Validation 的样本"

print("结果文件：", result_path)
print("已有结果：", len(completed_ids), "/ 1000")

new_count = 0

for sample in val_samples:
    if sample["sample_id"] in completed_ids:
        continue

    # 函数只会将 system 和 user 消息交给模型；
    # assistant 中的标准答案不会作为输入。
    prediction, seconds = zero_shot_predict(
        sample,
        max_new_tokens=384,
    )

    # 保留原始回答，包括可能出现的 Markdown 代码围栏。
    # 评测时才能真实统计格式错误。
    record = {
        "sample_id": sample["sample_id"],
        "task": sample["task"],
        "prediction": prediction,
        "latency_seconds": round(seconds, 4),
    }

    # 每生成一条就写入文件，减少运行中断造成的损失。
    with open(result_path, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
        file.flush()

    completed_ids.add(sample["sample_id"])
    new_count += 1

    if new_count % 25 == 0 or len(completed_ids) == len(val_samples):
        print(f"已完成：{len(completed_ids)}/1000；本次新增：{new_count}")

print("本次运行结束。结果文件：", result_path)

### 保存 Zero-shot 原始结果

原始 Qwen 已完成扩展 Validation 的 1,000 条回答。本步骤检查结果文件是否包含全部样本、是否有重复编号，然后下载备份。

保存的是模型的原始回答，包括可能出现的 JSON 代码围栏。后续评分会使用这些原始回答，不修改预测内容。

In [ ]:
import json
import hashlib
from collections import Counter
from google.colab import files

result_path = "/content/phase4_zero_shot_val_predictions.jsonl"

# 逐行回读，确认文件确实保存了全部预测。
with open(result_path, "rb") as file:
    file_bytes = file.read()

records = [
    json.loads(line)
    for line in file_bytes.decode("utf-8").splitlines()
    if line.strip()
]

saved_ids = [record["sample_id"] for record in records]
expected_ids = {sample["sample_id"] for sample in val_samples}

assert len(records) == 1000, f"实际保存 {len(records)} 条，并非 1000 条"
assert len(saved_ids) == len(set(saved_ids)), "结果文件中有重复 sample_id"
assert set(saved_ids) == expected_ids, "保存的样本与当前 Validation 不完全一致"

print("结果回读：通过")
print("总数：", len(records))
print("任务数量：", dict(Counter(record["task"] for record in records)))
print("SHA-256：", hashlib.sha256(file_bytes).hexdigest())
print("文件大小：", round(len(file_bytes) / 1024, 2), "KB")

# 下载到自己的电脑，避免 Colab 运行时结束后 /content 文件丢失。
files.download(result_path)

### Zero-shot：Task A 评分

**目的**：检查原始 Qwen 的 JSON 格式和字段提取是否正确。

| 指标 | 评分方式 |
|---|---|
| 严格 JSON | 原始回答必须能直接解析；代码围栏算失败 |
| 严格字段准确率 | 与标准答案比较；格式失败时，该条的 8 个字段均计为未通过 |
| `null` 准确率 | 只检查标准答案要求填 `null` 的字段 |
| 去围栏后字段准确率 | 仅作辅助分析，区分格式错误与内容错误 |

本步骤使用已保存的 500 条 Task A 回答，不重新运行模型。

In [ ]:
import json
import re

# 使用刚才回读通过的结果文件。
result_path = "/content/phase4_zero_shot_val_predictions.jsonl"
with open(result_path, "r", encoding="utf-8") as file:
    predictions = {
        row["sample_id"]: row
        for line in file if line.strip()
        for row in [json.loads(line)]
    }

a_samples = [sample for sample in val_samples if sample["task"] == "A"]
assert len(a_samples) == 500
assert all(sample["sample_id"] in predictions for sample in a_samples)

fields = [
    "load_id", "truck_id", "origin_city", "destination_city",
    "revenue_usd", "fuel_used_gallons",
    "scheduled_delivery", "actual_delivery",
]
text_fields = set(fields) - {"revenue_usd", "fuel_used_gallons"}

def parse_json(text, remove_fence=False):
    """解析回答；辅助评分时只允许去掉包住整个回答的代码围栏。"""
    text = text.strip()
    if remove_fence:
        match = re.fullmatch(
            r"```(?:json)?\s*\n?(.*?)\s*```",
            text,
            flags=re.IGNORECASE | re.DOTALL,
        )
        if match:
            text = match.group(1).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def schema_correct(answer):
    """检查八个字段是否齐全、没有多余字段，且值的类型合规。"""
    if not isinstance(answer, dict) or set(answer) != set(fields):
        return False

    for name in fields:
        value = answer[name]
        if value is None:
            continue
        if name in text_fields and not isinstance(value, str):
            return False
        if name not in text_fields and (
            isinstance(value, bool) or not isinstance(value, (int, float))
        ):
            return False
    return True

def field_correct(actual, expected):
    """比较单个字段的值，同时避免把 True 当成数字 1。"""
    if isinstance(actual, bool) or isinstance(expected, bool):
        return type(actual) is type(expected) and actual == expected
    return type(actual) is type(expected) and actual == expected

strict_json_count = strict_schema_count = strict_field_count = 0
clean_json_count = clean_field_count = 0
strict_null_count = clean_null_count = null_total = 0

for sample in a_samples:
    sample_id = sample["sample_id"]
    raw = predictions[sample_id]["prediction"]
    expected = json.loads(sample["messages"][-1]["content"])

    # 主指标：直接解析原始回答，不改动模型输出。
    strict = parse_json(raw)
    # 辅助指标：只去掉完整代码围栏，不补字段、不修改数值。
    cleaned = parse_json(raw, remove_fence=True)

    strict_json_count += isinstance(strict, dict)
    strict_schema_count += schema_correct(strict)
    clean_json_count += isinstance(cleaned, dict)

    for name in fields:
        truth = expected[name]
        if truth is None:
            null_total += 1

        # 无法解析或缺少字段时，主指标中的这个字段记为错误。
        if isinstance(strict, dict) and name in strict:
            correct = field_correct(strict[name], truth)
            strict_field_count += correct
            if truth is None:
                strict_null_count += correct

        if isinstance(cleaned, dict) and name in cleaned:
            correct = field_correct(cleaned[name], truth)
            clean_field_count += correct
            if truth is None:
                clean_null_count += correct

print(f"严格 JSON：{strict_json_count}/500 = {strict_json_count / 500:.2%}")
print(f"严格结构正确：{strict_schema_count}/500 = {strict_schema_count / 500:.2%}")
print(f"严格字段正确：{strict_field_count}/4000 = {strict_field_count / 4000:.2%}")
print(f"严格 null 正确：{strict_null_count}/{null_total}")

print("\n辅助分析（仅去完整代码围栏）：")
print(f"可解析 JSON 对象：{clean_json_count}/500 = {clean_json_count / 500:.2%}")
print(f"字段正确：{clean_field_count}/4000 = {clean_field_count / 4000:.2%}")
print(f"null 正确：{clean_null_count}/{null_total}")

### Zero-shot：Task B 评分

**目的**：检查原始 Qwen 对 500 条延误判断题的分类表现。

统计 Accuracy、两类标签的 Precision/Recall/F1、Macro-F1 和混淆矩阵。无法解析或标签不合规的回答计为错误；本步骤只读取已保存的预测，不重新运行模型。

In [ ]:
import json
from collections import Counter

result_path = "/content/phase4_zero_shot_val_predictions.jsonl"

with open(result_path, "r", encoding="utf-8") as file:
    predictions = {
        row["sample_id"]: row
        for line in file if line.strip()
        for row in [json.loads(line)]
    }

b_samples = [sample for sample in val_samples if sample["task"] == "B"]
assert len(b_samples) == 500
assert all(sample["sample_id"] in predictions for sample in b_samples)

# 行代表标准答案，列代表模型预测。
# "__invalid__" 表示无法得到合规的 delayed/on_time 标签。
confusion = Counter()
invalid_count = 0
error_ids = []

for sample in b_samples:
    sample_id = sample["sample_id"]
    truth = json.loads(sample["messages"][-1]["content"])["delay_label"]
    raw = predictions[sample_id]["prediction"].strip()

    try:
        answer = json.loads(raw)
        # 只接受恰好包含 delay_label、且标签合法的 JSON 对象。
        if (
            isinstance(answer, dict)
            and set(answer) == {"delay_label"}
            and answer["delay_label"] in {"delayed", "on_time"}
        ):
            predicted = answer["delay_label"]
        else:
            predicted = "__invalid__"
    except json.JSONDecodeError:
        predicted = "__invalid__"

    confusion[(truth, predicted)] += 1

    if predicted == "__invalid__":
        invalid_count += 1
    if predicted != truth:
        error_ids.append(sample_id)

def class_metrics(label):
    """按单个标签计算查准率、查全率和 F1。"""
    tp = confusion[(label, label)]
    fp = sum(
        confusion[(other, label)]
        for other in {"delayed", "on_time"} if other != label
    )
    fn = sum(
        confusion[(label, other)]
        for other in {"delayed", "on_time", "__invalid__"}
        if other != label
    )

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )
    return precision, recall, f1

correct = confusion[("delayed", "delayed")] + confusion[("on_time", "on_time")]
delayed_scores = class_metrics("delayed")
on_time_scores = class_metrics("on_time")
macro_f1 = (delayed_scores[2] + on_time_scores[2]) / 2

print(f"Task B 总数：{len(b_samples)}")
print(f"无法得到合法标签：{invalid_count}")
print(f"Accuracy：{correct}/500 = {correct / 500:.2%}")
print(f"delayed  Precision / Recall / F1："
      f"{delayed_scores[0]:.2%} / {delayed_scores[1]:.2%} / {delayed_scores[2]:.2%}")
print(f"on_time  Precision / Recall / F1："
      f"{on_time_scores[0]:.2%} / {on_time_scores[1]:.2%} / {on_time_scores[2]:.2%}")
print(f"Macro-F1：{macro_f1:.2%}")

print("\n混淆矩阵（行=标准答案，列=模型预测）：")
print("                 delayed  on_time  无效")
for truth in ("delayed", "on_time"):
    print(
        f"{truth:>10}       "
        f"{confusion[(truth, 'delayed')]:>7}  "
        f"{confusion[(truth, 'on_time')]:>7}  "
        f"{confusion[(truth, '__invalid__')]:>4}"
    )

print("\n错误总数：", len(error_ids))
print("错误样本 ID（前 10 个）：", error_ids[:10])

### Zero-shot 基线结果

| 任务 | 关键结果 | 主要问题 |
|---|---|---|
| Task A（500 条） | 严格 JSON 0.20%；严格字段 0.18%；去围栏后字段 93.97% | 大部分回答加了代码围栏；缺失值处理也有错误 |
| Task B（500 条） | Accuracy 66.80%；Macro-F1 40.05%；on_time Recall 0% | 几乎全部预测为 delayed，无法识别 on_time |

**结论**：原始 Qwen 已能提取不少字段内容，但未稳定遵守 JSON 格式；延误判断存在明显的单类预测倾向。这些是扩展 Validation 上的实测结果，后续将用同一批题目比较 Few-shot 和 Phase 3 LoRA。

## Phase 4.2.3｜Few-shot：选择示范题

**目的**：给原始 Qwen 少量已作答的例子，观察它能否更好地遵守 JSON 格式并判断延误。

**做法**：只从 Train 固定选择 4 条示范题：Task A 的完整记录和缺失记录各 1 条，Task B 的 delayed 和 on_time 各 1 条。检查示范题不属于 Val 或 Test，并记录样本编号，保证实验可复现。

Few-shot 仍使用原始 Qwen，不训练新模型。

In [ ]:
import json
import os
import zipfile

dataset_zip = "/content/FleetMind_LLM_Phase4_Evaluation_Dataset.zip"
assert os.path.exists(dataset_zip), "找不到 Phase 4 数据包"

with zipfile.ZipFile(dataset_zip, "r") as archive:
    # ZIP 内可能有文件夹前缀，因此按文件名定位。
    def read_jsonl(filename):
        matches = [
            name for name in archive.namelist()
            if os.path.basename(name) == filename
        ]
        assert len(matches) == 1, f"找不到唯一的 {filename}：{matches}"
        content = archive.read(matches[0]).decode("utf-8")
        return [
            json.loads(line)
            for line in content.splitlines()
            if line.strip()
        ]

    train_samples = read_jsonl("train.jsonl")
    test_samples = read_jsonl("test.jsonl")

assert len(train_samples) == 1600
assert len(val_samples) == 1000
assert len(test_samples) == 2000

train_ids = {sample["sample_id"] for sample in train_samples}
val_ids = {sample["sample_id"] for sample in val_samples}
test_ids = {sample["sample_id"] for sample in test_samples}

# 这里只用 Test 的编号检查集合是否重叠，不读取其答案做示范或评测。
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)

def first_by_id(samples):
    """按编号排序后取第一条，避免凭模型表现主观挑选。"""
    candidates = sorted(samples, key=lambda sample: sample["sample_id"])
    assert candidates, "找不到符合条件的 Train 示例"
    return candidates[0]

train_a = [sample for sample in train_samples if sample["task"] == "A"]
train_b = [sample for sample in train_samples if sample["task"] == "B"]

# Task A：一条完整记录，一条明确省略了字段的记录。
a_complete = first_by_id(
    [sample for sample in train_a if not sample.get("omitted_fields")]
)
a_missing = first_by_id(
    [sample for sample in train_a if sample.get("omitted_fields")]
)

# Task B：每个标签各一条，避免示范题只展示 delayed。
b_delayed = first_by_id([
    sample for sample in train_b
    if json.loads(sample["messages"][-1]["content"])["delay_label"] == "delayed"
])
b_on_time = first_by_id([
    sample for sample in train_b
    if json.loads(sample["messages"][-1]["content"])["delay_label"] == "on_time"
])

fewshot_examples = {
    "A": [a_complete, a_missing],
    "B": [b_delayed, b_on_time],
}

for task, examples in fewshot_examples.items():
    print(f"\nTask {task} 示范题：")
    for sample in examples:
        print(
            sample["sample_id"],
            "| 省略字段：", sample.get("omitted_fields", []),
            "| 标准答案：", sample["messages"][-1]["content"]
        )

selected_ids = {
    sample["sample_id"]
    for examples in fewshot_examples.values()
    for sample in examples
}
assert len(selected_ids) == 4
assert selected_ids <= train_ids
assert selected_ids.isdisjoint(val_ids | test_ids)

print("\n示范题全部来自 Train，与 Val/Test 无重叠：通过")

### Few-shot 流程试跑

将固定的 Train 示范题放在当前 Validation 题目前面，让原始 Qwen 参考其作答格式。Task A、B 各试跑 2 条，检查回答是否完整、能否直接解析为 JSON。试跑结果不计入正式指标。

In [ ]:
import json
import time
import torch

def few_shot_predict(sample, max_new_tokens=384):
    """用同一原始 Qwen，根据固定 Train 示例回答一条 Validation 题。"""
    task = sample["task"]
    examples = fewshot_examples[task]

    # 输入顺序：系统要求 → 两条 Train 示范问答 → 当前题目。
    # 当前 Validation 样本的 assistant 消息是标准答案，绝不放进输入。
    messages = [sample["messages"][0]]

    for example in examples:
        assert example["messages"][1]["role"] == "user"
        assert example["messages"][2]["role"] == "assistant"
        messages.extend(example["messages"][1:3])

    assert sample["messages"][1]["role"] == "user"
    messages.append(sample["messages"][1])

    # 与 Zero-shot 使用同一个 tokenizer、聊天模板和非思考模式。
    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    )
    model_inputs = {
        name: tensor.to(base_model.device)
        for name, tensor in model_inputs.items()
    }

    start = time.perf_counter()
    with torch.inference_mode():
        output_ids = base_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    seconds = time.perf_counter() - start

    # 只解码新生成的回答，不把示范题或当前题目算进回答。
    input_length = model_inputs["input_ids"].shape[-1]
    generated_ids = output_ids[0, input_length:]
    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return answer, seconds


# 与此前 Zero-shot 试跑一样，各取两条，仅检查流程。
pilot = (
    [sample for sample in val_samples if sample["task"] == "A"][:2]
    + [sample for sample in val_samples if sample["task"] == "B"][:2]
)

for sample in pilot:
    answer, seconds = few_shot_predict(sample)

    try:
        json.loads(answer)
        valid_json = True
    except json.JSONDecodeError:
        valid_json = False

    print(f"\n{sample['sample_id']} | Task {sample['task']} | {seconds:.2f} 秒")
    print("严格 JSON 有效：", valid_json)
    print("模型回答：", answer)

###Few-shot 全量验证

使用固定的 4 条 Train 示范题，在同一份 Validation 1,000 条上评估原始 Qwen。生成设置与 Zero-shot 一致：关闭 thinking、确定性生成、最多 384 个新 token。

逐条保存原始回答和耗时；中断后可从已保存的结果继续。完成后再统一计算指标。

In [ ]:
import json
import os
import hashlib
from collections import Counter

result_path = "/content/phase4_few_shot_val_predictions.jsonl"
config_path = "/content/phase4_few_shot_config.json"

# 记录本次实验使用的模型、示范题和 Validation 编号。
# 恢复运行时用它检查配置，避免把不同实验的结果写进同一文件。
experiment_config = {
    "model_id": MODEL_ID,
    "method": "few_shot",
    "max_new_tokens": 384,
    "enable_thinking": False,
    "do_sample": False,
    "example_ids": {
        task: [sample["sample_id"] for sample in examples]
        for task, examples in fewshot_examples.items()
    },
    "validation_ids_sha256": hashlib.sha256(
        "\n".join(sorted(sample["sample_id"] for sample in val_samples))
        .encode("utf-8")
    ).hexdigest(),
}

if os.path.exists(config_path):
    with open(config_path, "r", encoding="utf-8") as file:
        saved_config = json.load(file)
    assert saved_config == experiment_config, (
        "已有结果使用了不同配置，请先检查，不要混合两次实验。"
    )
else:
    assert not os.path.exists(result_path), (
        "已有预测文件但找不到配置文件，请先检查其来源。"
    )
    with open(config_path, "w", encoding="utf-8") as file:
        json.dump(experiment_config, file, ensure_ascii=False, indent=2)

# 核对当前确实是扩展后的 Validation。
task_counts = Counter(sample["task"] for sample in val_samples)
assert len(val_samples) == 1000
assert task_counts == {"A": 500, "B": 500}

valid_ids = {sample["sample_id"] for sample in val_samples}
completed_ids = set()

# 如果运行曾中断，读取已完成编号，重新运行时跳过它们。
if os.path.exists(result_path):
    with open(result_path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                record = json.loads(line)
                sample_id = record["sample_id"]
                assert sample_id not in completed_ids, f"重复结果：{sample_id}"
                completed_ids.add(sample_id)

assert completed_ids <= valid_ids, "已有结果不属于当前 Validation"

print("预测文件：", result_path)
print("实验配置：", config_path)
print("已有结果：", len(completed_ids), "/ 1000")

new_count = 0

for sample in val_samples:
    if sample["sample_id"] in completed_ids:
        continue

    # few_shot_predict 只加入固定的 Train 示范题和当前问题；
    # 当前 Validation 样本的标准答案不会交给模型。
    answer, seconds = few_shot_predict(sample, max_new_tokens=384)

    record = {
        "sample_id": sample["sample_id"],
        "task": sample["task"],
        "prediction": answer,
        "latency_seconds": round(seconds, 4),
    }

    # 每条结果立即写入文件，方便中断后继续。
    with open(result_path, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
        file.flush()

    completed_ids.add(sample["sample_id"])
    new_count += 1

    if new_count % 25 == 0 or len(completed_ids) == 1000:
        print(f"已完成：{len(completed_ids)}/1000；本次新增：{new_count}")

print("本次运行结束。预测文件：", result_path)

### Few-shot 结果检查与备份

核对 1,000 条预测是否齐全、编号是否重复、Task A/B 是否各 500 条。下载预测文件和实验配置，保存本次 Few-shot 的原始回答与示范题设置。

In [ ]:
import json
import hashlib
from collections import Counter
from google.colab import files

result_path = "/content/phase4_few_shot_val_predictions.jsonl"
config_path = "/content/phase4_few_shot_config.json"

# 回读完整文件，确认中断和恢复后没有漏题或重复题。
with open(result_path, "rb") as file:
    result_bytes = file.read()

records = [
    json.loads(line)
    for line in result_bytes.decode("utf-8").splitlines()
    if line.strip()
]
saved_ids = [row["sample_id"] for row in records]
val_ids = {sample["sample_id"] for sample in val_samples}
task_counts = Counter(row["task"] for row in records)

assert len(records) == 1000, f"结果只有 {len(records)} 条"
assert len(saved_ids) == len(set(saved_ids)), "存在重复 sample_id"
assert set(saved_ids) == val_ids, "预测编号与 Validation 不完全一致"
assert task_counts == {"A": 500, "B": 500}

with open(config_path, "r", encoding="utf-8") as file:
    saved_config = json.load(file)

assert saved_config == experiment_config, "保存的实验配置与当前配置不一致"

print("结果回读：通过")
print("总数：", len(records))
print("任务数量：", dict(task_counts))
print("预测文件 SHA-256：", hashlib.sha256(result_bytes).hexdigest())
print("示范题编号：", saved_config["example_ids"])

# 两个文件都要下载：预测内容与实验设置是一组完整证据。
files.download(result_path)
files.download(config_path)

### Few-shot：Task A 评分

使用与 Zero-shot 完全相同的严格 JSON、结构、字段和 `null` 评分规则。去围栏后的结果仍只作辅助分析；本步骤不重新运行模型。

In [ ]:
import json
import re

# 读取 Few-shot 的原始预测；不重新运行模型。
result_path = "/content/phase4_few_shot_val_predictions.jsonl"

with open(result_path, "r", encoding="utf-8") as file:
    predictions = {
        row["sample_id"]: row
        for line in file if line.strip()
        for row in [json.loads(line)]
    }

a_samples = [sample for sample in val_samples if sample["task"] == "A"]
assert len(a_samples) == 500
assert all(sample["sample_id"] in predictions for sample in a_samples)

fields = [
    "load_id", "truck_id", "origin_city", "destination_city",
    "revenue_usd", "fuel_used_gallons",
    "scheduled_delivery", "actual_delivery",
]
text_fields = set(fields) - {"revenue_usd", "fuel_used_gallons"}

def parse_json(text, remove_fence=False):
    """解析回答；辅助分析时只去掉包住整个回答的代码围栏。"""
    text = text.strip()

    if remove_fence:
        match = re.fullmatch(
            r"```(?:json)?\s*\n?(.*?)\s*```",
            text,
            flags=re.IGNORECASE | re.DOTALL,
        )
        if match:
            text = match.group(1).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def schema_correct(answer):
    """检查字段是否齐全、没有多余字段，且字段类型合规。"""
    if not isinstance(answer, dict) or set(answer) != set(fields):
        return False

    for name in fields:
        value = answer[name]

        # 输入中没有的信息允许输出 JSON null。
        if value is None:
            continue

        if name in text_fields and not isinstance(value, str):
            return False

        if name not in text_fields and (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
        ):
            return False

    return True

def field_correct(actual, expected):
    """比较单个字段，同时避免把布尔值当成数字。"""
    if isinstance(actual, bool) or isinstance(expected, bool):
        return type(actual) is type(expected) and actual == expected
    return type(actual) is type(expected) and actual == expected

strict_json_count = 0
strict_schema_count = 0
strict_field_count = 0
strict_null_count = 0

clean_json_count = 0
clean_field_count = 0
clean_null_count = 0
null_total = 0

for sample in a_samples:
    sample_id = sample["sample_id"]
    raw_answer = predictions[sample_id]["prediction"]
    expected = json.loads(sample["messages"][-1]["content"])

    # 主指标：直接解析模型原始回答，不删除围栏或修改内容。
    strict_answer = parse_json(raw_answer)

    # 辅助分析：只允许去掉包住整个回答的代码围栏。
    clean_answer = parse_json(raw_answer, remove_fence=True)

    strict_json_count += isinstance(strict_answer, dict)
    strict_schema_count += schema_correct(strict_answer)
    clean_json_count += isinstance(clean_answer, dict)

    for name in fields:
        truth = expected[name]

        if truth is None:
            null_total += 1

        # 原始回答无效或缺少字段时，严格字段计为错误。
        if isinstance(strict_answer, dict) and name in strict_answer:
            correct = field_correct(strict_answer[name], truth)
            strict_field_count += correct

            if truth is None:
                strict_null_count += correct

        # 辅助指标使用去围栏后的解析结果。
        if isinstance(clean_answer, dict) and name in clean_answer:
            correct = field_correct(clean_answer[name], truth)
            clean_field_count += correct

            if truth is None:
                clean_null_count += correct

print("===== Few-shot：Task A 严格指标 =====")
print(f"严格 JSON：{strict_json_count}/500 = {strict_json_count / 500:.2%}")
print(f"严格结构正确：{strict_schema_count}/500 = {strict_schema_count / 500:.2%}")
print(f"严格字段正确：{strict_field_count}/4000 = {strict_field_count / 4000:.2%}")
print(f"严格 null 正确：{strict_null_count}/{null_total}")

print("\n===== 辅助分析：仅去完整代码围栏 =====")
print(f"可解析 JSON 对象：{clean_json_count}/500 = {clean_json_count / 500:.2%}")
print(f"字段正确：{clean_field_count}/4000 = {clean_field_count / 4000:.2%}")
print(f"null 正确：{clean_null_count}/{null_total}")

### Few-shot：Task B 评分

在同一批 500 条延误判断题上，使用与 Zero-shot 相同的规则，统计 Accuracy、两类标签的 Precision/Recall/F1 和混淆矩阵。无效回答计为错误，不重新运行模型。

In [ ]:
import json
from collections import Counter

result_path = "/content/phase4_few_shot_val_predictions.jsonl"

with open(result_path, "r", encoding="utf-8") as file:
    predictions = {
        row["sample_id"]: row
        for line in file if line.strip()
        for row in [json.loads(line)]
    }

b_samples = [sample for sample in val_samples if sample["task"] == "B"]
assert len(b_samples) == 500
assert all(sample["sample_id"] in predictions for sample in b_samples)

# 行是标准答案，列是模型预测；无效回答单独计数。
confusion = Counter()
error_ids = []
invalid_count = 0

for sample in b_samples:
    sample_id = sample["sample_id"]
    truth = json.loads(sample["messages"][-1]["content"])["delay_label"]
    raw = predictions[sample_id]["prediction"].strip()

    try:
        answer = json.loads(raw)
        if (
            isinstance(answer, dict)
            and set(answer) == {"delay_label"}
            and answer["delay_label"] in {"delayed", "on_time"}
        ):
            predicted = answer["delay_label"]
        else:
            predicted = "__invalid__"
    except json.JSONDecodeError:
        predicted = "__invalid__"

    confusion[(truth, predicted)] += 1
    invalid_count += predicted == "__invalid__"

    if predicted != truth:
        error_ids.append(sample_id)

def class_metrics(label):
    """计算指定标签的 Precision、Recall 和 F1。"""
    tp = confusion[(label, label)]
    fp = sum(
        confusion[(other, label)]
        for other in ("delayed", "on_time")
        if other != label
    )
    fn = sum(
        confusion[(label, other)]
        for other in ("delayed", "on_time", "__invalid__")
        if other != label
    )

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )
    return precision, recall, f1

correct = confusion[("delayed", "delayed")] + confusion[("on_time", "on_time")]
delayed = class_metrics("delayed")
on_time = class_metrics("on_time")
macro_f1 = (delayed[2] + on_time[2]) / 2

print(f"Task B 总数：{len(b_samples)}")
print(f"无效回答：{invalid_count}")
print(f"Accuracy：{correct}/500 = {correct / 500:.2%}")
print(
    "delayed Precision / Recall / F1："
    f"{delayed[0]:.2%} / {delayed[1]:.2%} / {delayed[2]:.2%}"
)
print(
    "on_time Precision / Recall / F1："
    f"{on_time[0]:.2%} / {on_time[1]:.2%} / {on_time[2]:.2%}"
)
print(f"Macro-F1：{macro_f1:.2%}")

print("\n混淆矩阵（行=标准答案，列=模型预测）：")
print("                 delayed  on_time  无效")
for truth in ("delayed", "on_time"):
    print(
        f"{truth:>10}       "
        f"{confusion[(truth, 'delayed')]:>7}  "
        f"{confusion[(truth, 'on_time')]:>7}  "
        f"{confusion[(truth, '__invalid__')]:>4}"
    )

print("\n错误总数：", len(error_ids))
print("错误样本 ID（前 10 个）：", error_ids[:10])

### Few-shot 阶段结论

在相同的扩展 Validation 和评分规则下，Few-shot 将 Task A 严格 JSON 有效率提高到 100%，严格字段正确率达到 99.08%。Task B 的 on_time Recall 从 0% 提高到 30.30%，Macro-F1 从 40.05% 提高到 54.01%；但 Accuracy 从 66.80% 降至 62.20%。

这说明示范题改善了输出格式，并缓解了只预测 delayed 的倾向，但时间判断仍不稳定。下一步评测 Phase 3 已训练的 LoRA，使用同一批 Validation 题目。

## Phase 4.2.4｜恢复 Phase 3 最佳 LoRA

**目的**：使用 Phase 3 已训练好的最佳 adapter，在当前扩展 Validation 上重新评测。

本步骤从 Phase 3 备份包提取 `best_adapter`，核对基座模型及 LoRA 参数。使用已有模型，不重新训练。

In [ ]:
import json
import os
import zipfile
from pathlib import Path
from google.colab import files

phase3_zip = Path("/content/FleetMind_LLM_Phase3_Results.zip")
adapter_dir = Path("/content/phase3_best_adapter")

# Colab 运行时曾重置：若备份包已不在 /content，就从桌面重新上传。
if not phase3_zip.exists():
    print("请上传 FleetMind_LLM_Phase3_Results.zip")
    uploaded = files.upload()
    assert phase3_zip.name in uploaded, "未上传指定的 Phase 3 备份包"

adapter_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(phase3_zip, "r") as archive:
    # 只提取最终选出的 best_adapter，不恢复全部训练 checkpoint。
    members = [
        name for name in archive.namelist()
        if name.startswith("best_adapter/") and not name.endswith("/")
    ]
    assert members, "备份包中找不到 best_adapter"

    for name in members:
        relative = Path(name).relative_to("best_adapter")

        # 防止异常 ZIP 路径写到预期文件夹之外。
        assert ".." not in relative.parts
        target = adapter_dir / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))

config_file = adapter_dir / "adapter_config.json"
assert config_file.exists(), "缺少 adapter_config.json"
assert any(adapter_dir.glob("*.safetensors")), "缺少 adapter 权重文件"

with open(config_file, "r", encoding="utf-8") as file:
    adapter_config = json.load(file)

print("adapter 目录：", adapter_dir)
print("提取文件：", [str(p.relative_to(adapter_dir)) for p in adapter_dir.rglob("*") if p.is_file()])
print("基座模型：", adapter_config.get("base_model_name_or_path"))
print("r / alpha / dropout：",
      adapter_config.get("r"),
      adapter_config.get("lora_alpha"),
      adapter_config.get("lora_dropout"))
print("目标模块：", adapter_config.get("target_modules"))

assert adapter_config["base_model_name_or_path"] == MODEL_ID
assert adapter_config["r"] == 16
assert adapter_config["lora_alpha"] == 32
assert adapter_config["lora_dropout"] == 0.05
assert set(adapter_config["target_modules"]) == {
    "q_proj", "k_proj", "v_proj", "o_proj"
}

print("Phase 3 最佳 adapter 恢复和配置核对通过")

###LoRA 加载与试跑

将 Phase 3 的 `best_adapter` 加载到同一 Qwen 基座上。使用与 Zero-shot 相同的题目输入和生成设置，先试跑 Task A、B 各 2 条，确认回答完整、格式正常，再进行全量验证。

### LoRA 加载环境修复

当前 torchao 0.10.0 与 PEFT 不兼容，导致 Phase 3 adapter 无法加载。卸载旧版 torchao 并重启 Python 运行时，再恢复模型与数据变量。无需重新训练或重跑已完成的预测。

In [ ]:
%pip uninstall -y torchao

In [ ]:
import json
import time
import torch
from peft import PeftModel

# 前面已核对：adapter 对应的基座就是当前 MODEL_ID。
assert MODEL_ID == "Qwen/Qwen3-0.6B"
assert len(val_samples) == 1000
assert (adapter_dir / "adapter_config.json").exists()

# 将 Phase 3 已训练的权重接到当前基座上，仅用于推理。
# 此操作可能包装当前 base_model；此前 Zero-shot/Few-shot 预测已保存，不再调用它们。
lora_model = PeftModel.from_pretrained(
    base_model,
    str(adapter_dir),
    is_trainable=False,
)
lora_model.eval()

device = next(lora_model.parameters()).device
print("LoRA 已加载；设备：", device)
print("当前 adapter：", lora_model.active_adapters)

def lora_predict(sample, max_new_tokens=384):
    """只把当前题目的 system 和 user 交给 Phase 3 LoRA 模型。"""

    # assistant 消息是标准答案，评测时不能作为模型输入。
    prompt_messages = sample["messages"][:2]

    model_inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    )
    model_inputs = {
        name: tensor.to(device)
        for name, tensor in model_inputs.items()
    }
    input_length = model_inputs["input_ids"].shape[-1]

    start = time.perf_counter()
    with torch.inference_mode():
        output_ids = lora_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    seconds = time.perf_counter() - start

    # 只保留新生成的回答，不包含输入题目。
    answer = tokenizer.decode(
        output_ids[0, input_length:],
        skip_special_tokens=True,
    ).strip()
    return answer, seconds

pilot = (
    [sample for sample in val_samples if sample["task"] == "A"][:2]
    + [sample for sample in val_samples if sample["task"] == "B"][:2]
)

for sample in pilot:
    answer, seconds = lora_predict(sample)

    try:
        json.loads(answer)
        valid_json = True
    except json.JSONDecodeError:
        valid_json = False

    print(f"\n{sample['sample_id']} | Task {sample['task']} | {seconds:.2f} 秒")
    print("严格 JSON 有效：", valid_json)
    print("模型回答：", answer)

### LoRA 全量验证

使用 Phase 3 的最佳 LoRA，在与 Zero-shot、Few-shot 相同的 Validation 1,000 条上生成原始回答。关闭 thinking、确定性生成，最多生成 384 个新 token。

逐条保存预测和耗时，并记录模型、adapter 与数据编号；中断后可跳过已完成样本继续运行。Test 暂不参与评测。

In [ ]:
import json
import hashlib
from collections import Counter
from pathlib import Path
from google.colab import drive

# 将结果保存在 Drive；Colab 重置后仍可找回。
drive.mount("/content/drive")
save_dir = Path("/content/drive/MyDrive/FleetMind-LLM/phase4")
save_dir.mkdir(parents=True, exist_ok=True)

result_path = save_dir / "phase4_lora_val_predictions.jsonl"
config_path = save_dir / "phase4_lora_val_config.json"

assert len(val_samples) == 1000
assert Counter(sample["task"] for sample in val_samples) == {"A": 500, "B": 500}
assert "lora_predict" in globals(), "请先运行 LoRA 加载与试跑代码格"

# 为 adapter 权重生成指纹，确认中断恢复后仍用同一份模型。
weight_path = adapter_dir / "adapter_model.safetensors"
assert weight_path.exists()

with open(weight_path, "rb") as file:
    adapter_sha256 = hashlib.sha256(file.read()).hexdigest()

experiment_config = {
    "method": "phase3_best_lora",
    "model_id": MODEL_ID,
    "adapter_sha256": adapter_sha256,
    "max_new_tokens": 384,
    "enable_thinking": False,
    "do_sample": False,
    "validation_ids_sha256": hashlib.sha256(
        "\n".join(sorted(sample["sample_id"] for sample in val_samples))
        .encode("utf-8")
    ).hexdigest(),
}

# 恢复运行时，先核对已有配置，避免混合不同模型或数据的预测。
if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as file:
        assert json.load(file) == experiment_config, "已有实验配置不一致"
else:
    assert not result_path.exists(), "预测文件存在，但缺少配置文件"
    with open(config_path, "w", encoding="utf-8") as file:
        json.dump(experiment_config, file, ensure_ascii=False, indent=2)

valid_ids = {sample["sample_id"] for sample in val_samples}
completed_ids = set()

if result_path.exists():
    with open(result_path, "r", encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue
            row = json.loads(line)
            sample_id = row["sample_id"]
            assert sample_id not in completed_ids, f"重复结果：{sample_id}"
            completed_ids.add(sample_id)

assert completed_ids <= valid_ids, "已有结果不属于当前 Validation"

print("预测文件：", result_path)
print("配置文件：", config_path)
print("已有结果：", len(completed_ids), "/ 1000")

new_count = 0

for sample in val_samples:
    if sample["sample_id"] in completed_ids:
        continue

    # lora_predict 只将当前题目的 system 和 user 交给模型。
    answer, seconds = lora_predict(sample, max_new_tokens=384)

    row = {
        "sample_id": sample["sample_id"],
        "task": sample["task"],
        "prediction": answer,  # 保留未经修改的原始回答
        "latency_seconds": round(seconds, 4),
    }

    # 每条回答写入 Drive；下次运行会从已保存的编号之后继续。
    with open(result_path, "a", encoding="utf-8") as file:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")
        file.flush()

    completed_ids.add(sample["sample_id"])
    new_count += 1

    if new_count % 25 == 0 or len(completed_ids) == 1000:
        print(f"已完成：{len(completed_ids)}/1000；本次新增：{new_count}")

print("本次运行结束。预测文件：", result_path)

### LoRA 结果完整性检查

回读 Drive 中的 LoRA 预测，确认 Validation 1,000 条全部齐全、没有重复，Task A/B 各 500 条。记录文件指纹，随后使用与前两种方法相同的规则评分。

In [ ]:
import json
import hashlib
from collections import Counter
from pathlib import Path

result_path = Path(
    "/content/drive/MyDrive/FleetMind-LLM/phase4/"
    "phase4_lora_val_predictions.jsonl"
)
config_path = result_path.with_name("phase4_lora_val_config.json")

with open(result_path, "rb") as file:
    result_bytes = file.read()

rows = [
    json.loads(line)
    for line in result_bytes.decode("utf-8").splitlines()
    if line.strip()
]
saved_ids = [row["sample_id"] for row in rows]
expected_ids = {sample["sample_id"] for sample in val_samples}
counts = Counter(row["task"] for row in rows)

assert len(rows) == 1000
assert len(saved_ids) == len(set(saved_ids)), "存在重复 sample_id"
assert set(saved_ids) == expected_ids, "与当前 Validation 编号不一致"
assert counts == {"A": 500, "B": 500}

with open(config_path, "r", encoding="utf-8") as file:
    config = json.load(file)

assert config["method"] == "phase3_best_lora"
assert config["model_id"] == "Qwen/Qwen3-0.6B"
assert config["max_new_tokens"] == 384

print("LoRA 结果回读：通过")
print("总数：", len(rows))
print("任务数量：", dict(counts))
print("SHA-256：", hashlib.sha256(result_bytes).hexdigest())
print("预测文件：", result_path)

### LoRA：Task A 评分

在扩展 Validation 的 500 条 Task A 上，统计严格 JSON、结构、字段和 `null` 准确率。格式失败的回答，其字段计为未通过；去围栏后的结果仅作辅助分析，与前两种方法使用相同评分口径。

In [ ]:
import json
import re
from pathlib import Path

result_path = Path(
    "/content/drive/MyDrive/FleetMind-LLM/phase4/"
    "phase4_lora_val_predictions.jsonl"
)

with open(result_path, "r", encoding="utf-8") as file:
    predictions = {
        row["sample_id"]: row
        for line in file if line.strip()
        for row in [json.loads(line)]
    }

a_samples = [sample for sample in val_samples if sample["task"] == "A"]
assert len(a_samples) == 500
assert all(sample["sample_id"] in predictions for sample in a_samples)

fields = [
    "load_id", "truck_id", "origin_city", "destination_city",
    "revenue_usd", "fuel_used_gallons",
    "scheduled_delivery", "actual_delivery",
]
text_fields = set(fields) - {"revenue_usd", "fuel_used_gallons"}

def parse_json(text, remove_fence=False):
    """解析原始回答；辅助分析时仅去掉包住整个回答的围栏。"""
    text = text.strip()

    if remove_fence:
        match = re.fullmatch(
            r"```(?:json)?\s*\n?(.*?)\s*```",
            text,
            flags=re.IGNORECASE | re.DOTALL,
        )
        if match:
            text = match.group(1).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def schema_correct(answer):
    """检查八个字段是否齐全、无多余字段且类型合规。"""
    if not isinstance(answer, dict) or set(answer) != set(fields):
        return False

    for name in fields:
        value = answer[name]
        if value is None:
            continue
        if name in text_fields and not isinstance(value, str):
            return False
        if name not in text_fields and (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
        ):
            return False
    return True

def field_correct(actual, expected):
    """值和类型均与标准答案相同才算正确。"""
    return type(actual) is type(expected) and actual == expected

strict_json_count = strict_schema_count = strict_field_count = 0
clean_json_count = clean_field_count = 0
strict_null_count = clean_null_count = null_total = 0

for sample in a_samples:
    sample_id = sample["sample_id"]
    raw = predictions[sample_id]["prediction"]
    expected = json.loads(sample["messages"][-1]["content"])

    # 主指标直接检查原始回答；辅助指标只允许去完整围栏。
    strict_answer = parse_json(raw)
    clean_answer = parse_json(raw, remove_fence=True)

    strict_json_count += isinstance(strict_answer, dict)
    strict_schema_count += schema_correct(strict_answer)
    clean_json_count += isinstance(clean_answer, dict)

    for name in fields:
        truth = expected[name]
        if truth is None:
            null_total += 1

        if isinstance(strict_answer, dict) and name in strict_answer:
            correct = field_correct(strict_answer[name], truth)
            strict_field_count += correct
            if truth is None:
                strict_null_count += correct

        if isinstance(clean_answer, dict) and name in clean_answer:
            correct = field_correct(clean_answer[name], truth)
            clean_field_count += correct
            if truth is None:
                clean_null_count += correct

print("===== LoRA：Task A 严格指标 =====")
print(f"严格 JSON：{strict_json_count}/500 = {strict_json_count / 500:.2%}")
print(f"严格结构正确：{strict_schema_count}/500 = {strict_schema_count / 500:.2%}")
print(f"严格字段正确：{strict_field_count}/4000 = {strict_field_count / 4000:.2%}")
print(f"严格 null 正确：{strict_null_count}/{null_total}")

print("\n===== 辅助分析：仅去完整代码围栏 =====")
print(f"可解析 JSON 对象：{clean_json_count}/500 = {clean_json_count / 500:.2%}")
print(f"字段正确：{clean_field_count}/4000 = {clean_field_count / 4000:.2%}")
print(f"null 正确：{clean_null_count}/{null_total}")

### LoRA：Task B 评分

在扩展 Validation 的 500 条延误判断题上，使用与 Zero-shot、Few-shot 相同的规则，统计 Accuracy、两类标签的 Precision/Recall/F1 和混淆矩阵。无效回答计为错误，不重新运行模型。

In [ ]:
import json
from collections import Counter
from pathlib import Path

result_path = Path(
    "/content/drive/MyDrive/FleetMind-LLM/phase4/"
    "phase4_lora_val_predictions.jsonl"
)

with open(result_path, "r", encoding="utf-8") as file:
    predictions = {
        row["sample_id"]: row
        for line in file if line.strip()
        for row in [json.loads(line)]
    }

b_samples = [sample for sample in val_samples if sample["task"] == "B"]
assert len(b_samples) == 500
assert all(sample["sample_id"] in predictions for sample in b_samples)

# 行表示标准答案，列表示模型预测。
confusion = Counter()
error_ids = []
invalid_count = 0

for sample in b_samples:
    sample_id = sample["sample_id"]
    truth = json.loads(sample["messages"][-1]["content"])["delay_label"]
    raw = predictions[sample_id]["prediction"].strip()

    try:
        answer = json.loads(raw)
        if (
            isinstance(answer, dict)
            and set(answer) == {"delay_label"}
            and answer["delay_label"] in {"delayed", "on_time"}
        ):
            predicted = answer["delay_label"]
        else:
            predicted = "__invalid__"
    except json.JSONDecodeError:
        predicted = "__invalid__"

    confusion[(truth, predicted)] += 1
    invalid_count += predicted == "__invalid__"

    if predicted != truth:
        error_ids.append(sample_id)

def class_metrics(label):
    """按一个标签计算 Precision、Recall 和 F1。"""
    tp = confusion[(label, label)]
    fp = sum(
        confusion[(other, label)]
        for other in ("delayed", "on_time")
        if other != label
    )
    fn = sum(
        confusion[(label, other)]
        for other in ("delayed", "on_time", "__invalid__")
        if other != label
    )

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )
    return precision, recall, f1

correct = confusion[("delayed", "delayed")] + confusion[("on_time", "on_time")]
delayed = class_metrics("delayed")
on_time = class_metrics("on_time")
macro_f1 = (delayed[2] + on_time[2]) / 2

print(f"Task B 总数：{len(b_samples)}")
print(f"无效回答：{invalid_count}")
print(f"Accuracy：{correct}/500 = {correct / 500:.2%}")
print(
    "delayed Precision / Recall / F1："
    f"{delayed[0]:.2%} / {delayed[1]:.2%} / {delayed[2]:.2%}"
)
print(
    "on_time Precision / Recall / F1："
    f"{on_time[0]:.2%} / {on_time[1]:.2%} / {on_time[2]:.2%}"
)
print(f"Macro-F1：{macro_f1:.2%}")

print("\n混淆矩阵（行=标准答案，列=模型预测）：")
print("                 delayed  on_time  无效")
for truth in ("delayed", "on_time"):
    print(
        f"{truth:>10}       "
        f"{confusion[(truth, 'delayed')]:>7}  "
        f"{confusion[(truth, 'on_time')]:>7}  "
        f"{confusion[(truth, '__invalid__')]:>4}"
    )

print("\n错误总数：", len(error_ids))
print("错误样本 ID（前 10 个）：", error_ids[:10])

### 四种方法在同一验证集上的对比

评测使用扩展后的 Validation：Task A 500 条、Task B 500 条。规则方法只用于 Task B；Zero-shot、Few-shot 和 LoRA 均使用 Qwen3-0.6B 基座，LoRA 加载 Phase 3 训练的最佳 adapter。

### Task A：物流信息提取

| 指标 | 规则方法 | Zero-shot | Few-shot | Phase 3 LoRA |
|---|---:|---:|---:|---:|
| 严格 JSON 格式正确 | 不适用 | 1/500（0.20%） | 500/500（100%） | **500/500（100%）** |
| 严格结构正确 | 不适用 | 1/500（0.20%） | 500/500（100%） | **500/500（100%）** |
| 严格字段正确 | 不适用 | 7/4000（0.18%） | 3963/4000（99.08%） | **4000/4000（100%）** |
| 严格 `null` 正确 | 不适用 | 2/197 | 161/197 | **197/197** |

> **格式说明**：Zero-shot 经常把答案放进 Markdown 代码围栏。仅移除完整围栏后，它的 500 条答案都能解析为 JSON，字段正确率为 3759/4000（93.97%）。上表仍按原始输出严格评分，不替模型修改答案。

### Task B：交付延误判断

| 指标 | 规则方法 | Zero-shot | Few-shot | Phase 3 LoRA |
|---|---:|---:|---:|---:|
| Accuracy | 500/500（100%） | 334/500（66.80%） | 311/500（62.20%） | **497/500（99.40%）** |
| Macro-F1 | 100% | 40.05% | 54.01% | **99.32%** |
| `on_time` Recall | 100% | 0% | 30.30% | **98.18%** |
| 无效回答 | 0 | 1 | 0 | 0 |

### 结论

- **LoRA 综合表现最好**：Task A 的格式、字段和 `null` 全部正确；Task B 只错 3 条。
- **Few-shot 改善了输出格式**，也让模型开始识别 `on_time`，但 Task B Accuracy 从 Zero-shot 的 66.80% 降至 62.20%。不能说 Few-shot 在所有指标上都更好。
- **规则方法在 Task B 达到 100%**，因为该任务的标签可由计划时间与实际时间直接计算。它是确定性参考方法，不属于训练后的模型成绩。
- 以上都是 **Validation 结果**；封存的 Test 集尚未用于这次方法对比。

## Phase 4.3｜分析 LoRA 的 3 条延误判断错误

### 4.3.1｜核对 LoRA 的延误判断错误

**目的**：找出 LoRA 在扩展 Validation 上错判的 3 条记录，观察它们是否具有共同特征。

**做法**：读取已经保存的预测文件，与 Validation 标准答案逐条比较；从题目中提取计划和实际交付时间，计算时间差。

**判定规则**：时间差 = 实际交付时间 − 计划交付时间。大于 0 秒为 `delayed`；小于或等于 0 秒为 `on_time`。本步骤只分析错误，不修改模型或标签。

In [ ]:
import json
import re
from datetime import datetime
from pathlib import Path

# 读取已经生成的 LoRA 预测；本步骤不会再次调用模型。
prediction_path = Path(
    "/content/drive/MyDrive/FleetMind-LLM/phase4/"
    "phase4_lora_val_predictions.jsonl"
)
assert prediction_path.is_file(), f"找不到预测文件：{prediction_path}"
assert "val_samples" in globals(), "请先运行加载扩展版 Validation 的单元格。"

with prediction_path.open("r", encoding="utf-8") as file:
    predictions = [json.loads(line) for line in file if line.strip()]

# 用 sample_id 对齐题目和预测，避免两个文件的行顺序不同造成错配。
val_by_id = {sample["sample_id"]: sample for sample in val_samples}
pred_by_id = {item["sample_id"]: item for item in predictions}

assert len(val_samples) == 1000 and len(val_by_id) == 1000, "Validation 应为 1000 条且编号不重复。"
assert len(predictions) == 1000 and len(pred_by_id) == 1000, "预测文件应为 1000 条且编号不重复。"
assert set(val_by_id) == set(pred_by_id), "预测编号与 Validation 编号不一致。"

# 标准答案和模型回答都是 JSON 字符串；解析后只比较 delay_label。
errors = []
for sample_id, sample in val_by_id.items():
    if sample["task"] != "B":
        continue

    expected = json.loads(sample["messages"][-1]["content"])["delay_label"]
    raw_prediction = pred_by_id[sample_id]["prediction"]

    try:
        predicted = json.loads(raw_prediction)["delay_label"]
    except (json.JSONDecodeError, KeyError, TypeError):
        predicted = "无效回答"

    if predicted != expected:
        errors.append((sample_id, sample, expected, predicted))

assert len(errors) == 3, f"预期有 3 条错误，实际找到 {len(errors)} 条。"

# 题目中应有两个时间：先是计划交付时间，再是实际交付时间。
# 同时打印原题，方便人工复核时间顺序和字段含义。
time_pattern = re.compile(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}(?:\.\d+)?")

print("LoRA Task B 错误数：", len(errors))

for sample_id, sample, expected, predicted in errors:
    question = sample["messages"][1]["content"]
    timestamps = time_pattern.findall(question)

    print(f"\n===== {sample_id} =====")
    print("原始题目：", question)
    print("标准标签：", expected)
    print("模型预测：", predicted)

    if len(timestamps) != 2:
        print(f"时间解析待人工核对：找到 {len(timestamps)} 个时间：{timestamps}")
        continue

    scheduled = datetime.fromisoformat(timestamps[0])
    actual = datetime.fromisoformat(timestamps[1])
    difference_seconds = (actual - scheduled).total_seconds()

    print("计划交付：", timestamps[0])
    print("实际交付：", timestamps[1])
    print(f"时间差（实际 − 计划）：{difference_seconds:+.6f} 秒")
    print("按时间重新计算的标签：",
          "delayed" if difference_seconds > 0 else "on_time")

### LoRA 错误分析结果

| 样本 ID | 实际交付比计划时间 | 正确标签 | LoRA 预测 | 观察 |
|---|---:|---|---|---|
| `LOAD00019804_B` | 提前 0.580845 秒 | `on_time` | `delayed` | 亚秒级边界错误 |
| `LOAD00009751_B` | 提前约 1 小时 2 分 | `on_time` | `delayed` | 明显提前，仍被错判 |
| `LOAD00019219_B` | 提前约 2 小时 11 分 | `on_time` | `delayed` | 明显提前，仍被错判 |

**结论**：3 条错误都是将 `on_time` 错判为 `delayed`。Validation 中 335 条 `delayed` 全部判对，165 条 `on_time` 判对 162 条。标准标签经时间差复核无误；模型仍存在少量偏向预测 `delayed` 的情况，但仅凭这 3 条不能确定具体原因。

### 4.3.2｜Few-shot 的错误变化

**目的**：解释为什么 Few-shot 的 Task B Accuracy 为 62.20%，低于 Zero-shot 的 66.80%。

**方法**：按 `sample_id` 对齐两种方法在同一批 500 道题上的回答，统计“答错变答对”和“答对变答错”的数量，并查看代表性题目。示范题改善了部分表现，并不保证每道题都会更准确。

In [ ]:
from pathlib import Path
from google.colab import files

# 当前运行时缺少 Zero-shot 预测文件，从电脑上传之前保存的结果。
target = Path("/content/phase4_zero_shot_val_predictions.jsonl")

if not target.is_file():
    print("请选择 phase4_zero_shot_val_predictions.jsonl")
    uploaded = files.upload()
    assert target.name in uploaded, f"请选择正确文件：{target.name}"

print("Zero-shot 文件已就绪：", target)
print("文件大小：", round(target.stat().st_size / 1024, 2), "KB")

In [ ]:
import json
import re
from collections import defaultdict
from datetime import datetime
from pathlib import Path

# 读取两份已有预测；如果路径不存在，先确认文件是否还在当前 Colab。
zero_path = Path("/content/phase4_zero_shot_val_predictions.jsonl")
few_path = Path("/content/phase4_few_shot_val_predictions.jsonl")

for path in (zero_path, few_path):
    assert path.is_file(), f"找不到文件：{path}"

assert "val_samples" in globals(), "请先运行加载扩展版 Validation 的单元格。"

def read_predictions(path):
    """按样本编号保存预测，避免依赖文件的行顺序。"""
    with path.open("r", encoding="utf-8") as file:
        rows = [json.loads(line) for line in file if line.strip()]
    result = {row["sample_id"]: row["prediction"] for row in rows}
    assert len(rows) == len(result) == 1000, f"{path.name} 数量或编号有误"
    return result

def get_label(answer):
    """无法解析的回答记为无效，不擅自修正模型输出。"""
    try:
        label = json.loads(answer)["delay_label"]
        return label if label in {"delayed", "on_time"} else "无效"
    except (json.JSONDecodeError, KeyError, TypeError):
        return "无效"

zero_predictions = read_predictions(zero_path)
few_predictions = read_predictions(few_path)
task_b = [s for s in val_samples if s["task"] == "B"]
assert len(task_b) == 500, "Task B 应有 500 条"

groups = defaultdict(list)

for sample in task_b:
    sample_id = sample["sample_id"]
    truth = get_label(sample["messages"][-1]["content"])
    zero = get_label(zero_predictions[sample_id])
    few = get_label(few_predictions[sample_id])

    # 用同一条标准答案分别判断两种方法是否正确。
    if zero == truth and few == truth:
        group = "两者都正确"
    elif zero == truth:
        group = "Zero-shot 对 → Few-shot 错"
    elif few == truth:
        group = "Zero-shot 错 → Few-shot 对"
    else:
        group = "两者都错误"

    groups[group].append((sample, truth, zero, few))

print("===== 同题对比：Task B 500 条 =====")
for name in (
    "两者都正确",
    "Zero-shot 对 → Few-shot 错",
    "Zero-shot 错 → Few-shot 对",
    "两者都错误",
):
    print(f"{name}：{len(groups[name])}")


time_pattern = re.compile(
    r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}(?:\.\d+)?"
)


In [ ]:
from datetime import datetime

def read_task_b_times(question):
    """根据时间前的文字识别计划和实际时间，不依赖出现顺序。"""
    found = {}

    for match in time_pattern.finditer(question):
        # 只看当前时间所在的小句，判断它描述的是哪一种时间。
        start = match.start()
        clause_start = max(
            question.rfind("；", 0, start),
            question.rfind(";", 0, start),
            question.rfind("\n", 0, start),
            question.rfind("。", 0, start),
        )
        clause = question[clause_start + 1:start]

        actual_pos = clause.rfind("实际")
        planned_pos = clause.rfind("计划")

        if actual_pos > planned_pos:
            key = "actual"
        elif planned_pos > actual_pos:
            key = "scheduled"
        else:
            return None  # 无法可靠识别，留待人工检查

        if key in found:
            return None  # 同一类时间出现多次，也留待人工检查
        found[key] = datetime.fromisoformat(match.group())

    return found if set(found) == {"scheduled", "actual"} else None


unresolved = []
label_mismatches = []

for sample in task_b:
    question = sample["messages"][1]["content"]
    times = read_task_b_times(question)

    if times is None:
        unresolved.append(sample["sample_id"])
        continue

    delta = (times["actual"] - times["scheduled"]).total_seconds()
    calculated = "delayed" if delta > 0 else "on_time"
    expected = get_label(sample["messages"][-1]["content"])

    if calculated != expected:
        label_mismatches.append(sample["sample_id"])

print("Task B 总数：", len(task_b))
print("无法可靠识别时间：", len(unresolved), unresolved[:10])
print("时间计算与标准标签不一致：", len(label_mismatches), label_mismatches[:10])

print("\n修正后的四条例题：")
for name in (
    "Zero-shot 对 → Few-shot 错",
    "Zero-shot 错 → Few-shot 对",
):
    for sample, truth, zero, few in groups[name][:2]:
        times = read_task_b_times(sample["messages"][1]["content"])
        if times is None:
            print(sample["sample_id"], "需人工核对时间")
            continue

        delta = (times["actual"] - times["scheduled"]).total_seconds()
        print(
            f"{sample['sample_id']} | 正确：{truth} | "
            f"Zero-shot：{zero} | Few-shot：{few} | "
            f"实际−计划：{delta:+.6f} 秒"
        )

### Few-shot 错误分析结果

| 同一道题的结果变化 | 数量 |
|---|---:|
| Zero-shot、Few-shot 都正确 | 260 |
| Zero-shot 正确 → Few-shot 错误 | 74 |
| Zero-shot 错误 → Few-shot 正确 | 51 |
| 两者都错误 | 115 |

Few-shot 修正了 **51** 道错题，但使 **74** 道原本答对的题变错，净减少 **23** 道正确答案。因此，Task B Accuracy 从 Zero-shot 的 **334/500（66.80%）** 降至 Few-shot 的 **311/500（62.20%）**。

例题显示两种变化都存在：Few-shot 能纠正提前 **0.580845 秒**的边界样本，也会将延误约 **4 小时 51 分**的样本误判为 `on_time`。500 条题目的时间解析及标准标签复核均通过，不能将准确率下降归因于标签错误。

## Phase 4.3｜错误分析总结

| 检查内容 | 结果 | 主要发现 |
|---|---:|---|
| LoRA Task B | 497/500 正确 | 3 条错误均将 `on_time` 判为 `delayed`；其中 1 条提前不足 1 秒，另 2 条分别提前约 1 小时和 2 小时 |
| Few-shot 与 Zero-shot 同题对比 | Few-shot 净减少 23 条正确答案 | Few-shot 修正 51 条错题，也使 74 条原本正确的题变错 |
| Task B 标签复核 | 500/500 一致 | 按题目中的计划与实际时间重新计算，均与标准标签一致 |
| Task A 字段 | LoRA 4000/4000；Few-shot 3963/4000 | Few-shot 的 37 次字段错误中，36 次涉及应输出 `null` 的字段 |

**结论**：LoRA 在扩展 Validation 上表现较好，但延误判断仍有少量误判，且并非都发生在时间边界。Few-shot 改善了部分题目，却没有提升 Task B 的总体准确率。上述分析描述了错误现象，暂不足以确定模型出错的根本原因。

## Phase 4.4｜独立困难样本检查

**目的**：检查模型在容易出错的表达和时间条件下是否仍能正确回答。

| 任务 | 计划样本数 | 重点检查 |
|---|---:|---|
| Task A：信息提取 | 50 | 字段缺失、字段顺序变化、正确输出 `null` |
| Task B：延误判断 | 50 | 实际时间先出现、跨日期、刚好准时、相差不足 1 秒 |

样本使用新编的业务编号和人工构造的内容，不从 Train、Validation 或 Test 复制题目。先检查样本与答案，再用现有 LoRA 评测；结果单独报告，不与正式 Validation 或最终 Test 成绩混合。

In [ ]:
import json
import re
import hashlib
from collections import Counter
from datetime import datetime, timedelta
from pathlib import Path

# 使用独立的合成编号；不复制 Train、Validation 或 Test 的业务记录。
system_text = (
    "你是物流业务数据助手。只依据用户提供的记录完成任务，不编造信息。"
    "只输出合法 JSON，不添加解释或 Markdown 代码围栏。"
)
fields = [
    "load_id", "truck_id", "origin_city", "destination_city",
    "revenue_usd", "fuel_used_gallons",
    "scheduled_delivery", "actual_delivery",
]
names = {
    "load_id": "运单编号", "truck_id": "车辆编号",
    "origin_city": "出发城市", "destination_city": "目的城市",
    "revenue_usd": "运单收入（美元）",
    "fuel_used_gallons": "实际耗油量（加仑）",
    "scheduled_delivery": "计划交付时间",
    "actual_delivery": "实际交付时间",
}
cities = ["Houston", "Seattle", "Atlanta", "Detroit", "Dallas"]
challenge_samples = []

# Task A：50 条。轮换字段顺序，并在部分题目中主动省略字段。
for i in range(50):
    scheduled = datetime(2025, 1, 1, 9, 15) + timedelta(days=i)
    record = {
        "load_id": f"SYNLOAD{i:05d}",
        "truck_id": f"SYNTRK{i % 12:03d}",
        "origin_city": cities[i % 5],
        "destination_city": cities[(i + 2) % 5],
        "revenue_usd": round(1200 + i * 17.35, 2),
        "fuel_used_gallons": round(80 + i * 1.7, 1),
        "scheduled_delivery": str(scheduled),
        "actual_delivery": str(scheduled + timedelta(minutes=i % 7 - 3)),
    }

    # 0、1、2 个字段缺失交替出现；缺失信息的答案必须是 null。
    missing = (
        [] if i % 3 == 0
        else [fields[(i % 7) + 1]] if i % 3 == 1
        else [fields[(i % 7) + 1], fields[((i + 3) % 7) + 1]]
    )
    shown_order = fields[i % 8:] + fields[:i % 8]
    details = "；".join(
        f"{names[field]}：{record[field]}"
        for field in shown_order if field not in missing
    )
    question = (
        "提取以下字段："
        + ", ".join(fields)
        + "。编号、城市和时间使用字符串，收入与耗油量使用数字。"
        "输入未提供的信息填 null，不要推测。只输出 JSON。\n\n"
        "运输资料：" + details + "。"
    )
    answer = {field: None if field in missing else record[field] for field in fields}

    challenge_samples.append({
        "sample_id": f"CHALLENGE_A_{i:03d}",
        "task": "A",
        "split": "challenge",
        "omitted_fields": missing,
        "messages": [
            {"role": "system", "content": system_text},
            {"role": "user", "content": question},
            {"role": "assistant", "content": json.dumps(answer, ensure_ascii=False)},
        ],
    })

# Task B：50 条。覆盖提前、准时、延误、跨日和微秒级时间差。
offsets = [-7200, -3600, -0.5, 0, 0.5, 3600, 7200, 0.000001, -0.000001, 60]
for i in range(50):
    scheduled = datetime(2025, 3, 1, 23, 59, 59, 750000) + timedelta(days=i)
    actual = scheduled + timedelta(seconds=offsets[i % len(offsets)])

    # 一半题目先写实际时间，避免模型或检查代码依赖固定顺序。
    parts = [
        f"计划时间：{scheduled}",
        f"实际时间：{actual}",
    ]
    if i % 2:
        parts.reverse()

    question = (
        "判断是否发生交付延误。实际交付时间晚于计划时间标为 delayed，"
        '否则标为 on_time。只输出 {"delay_label": "标签"}。\n\n'
        "交付记录：" + "；".join(parts) + "。"
    )
    answer = {"delay_label": "delayed" if actual > scheduled else "on_time"}

    challenge_samples.append({
        "sample_id": f"CHALLENGE_B_{i:03d}",
        "task": "B",
        "split": "challenge",
        "messages": [
            {"role": "system", "content": system_text},
            {"role": "user", "content": question},
            {"role": "assistant", "content": json.dumps(answer, ensure_ascii=False)},
        ],
    })

# 保存前检查：数量、编号、字段和 Task B 时间标签。
ids = [s["sample_id"] for s in challenge_samples]
assert len(ids) == len(set(ids)) == 100
assert Counter(s["task"] for s in challenge_samples) == {"A": 50, "B": 50}

if "val_samples" in globals():
    assert not (set(ids) & {s["sample_id"] for s in val_samples})

timestamp = r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{6})"
for sample in challenge_samples:
    question = sample["messages"][1]["content"]
    answer = json.loads(sample["messages"][2]["content"])

    if sample["task"] == "A":
        assert list(answer) == fields
        for field in fields:
            assert (answer[field] is None) == (field in sample["omitted_fields"])
    else:
        # 按文字标签分别提取时间，不依赖两个时间在题目里的顺序。
        planned = re.search(r"计划时间：" + timestamp, question)
        delivered = re.search(r"实际时间：" + timestamp, question)
        assert planned and delivered
        correct = (
            "delayed"
            if datetime.fromisoformat(delivered.group(1))
               > datetime.fromisoformat(planned.group(1))
            else "on_time"
        )
        assert answer["delay_label"] == correct

# 有 Drive 就存入 Drive；否则先存 /content，运行时结束前需下载。
drive_dir = Path("/content/drive/MyDrive/FleetMind-LLM/phase4")
save_dir = drive_dir if drive_dir.parent.is_dir() else Path("/content")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "phase4_challenge_100.jsonl"

content = "".join(
    json.dumps(s, ensure_ascii=False) + "\n" for s in challenge_samples
).encode("utf-8")

# 若已有同名文件，只允许内容完全一致，防止覆盖另一版困难样本。
if save_path.exists():
    assert save_path.read_bytes() == content, f"已有不同版本，请先核对：{save_path}"
else:
    save_path.write_bytes(content)

print("样本数量：", len(challenge_samples), dict(Counter(s["task"] for s in challenge_samples)))
print("Task B 标签：", dict(Counter(
    json.loads(s["messages"][2]["content"])["delay_label"]
    for s in challenge_samples if s["task"] == "B"
)))
print("数量、编号、答案与时间标签检查：通过")
print("保存位置：", save_path)
print("SHA-256：", hashlib.sha256(content).hexdigest())
print("\nTask A 示例：", challenge_samples[1]["messages"][1]["content"])
print("Task B 示例：", challenge_samples[51]["messages"][1]["content"])

### LoRA 困难样本评测

使用 Phase 3 最佳 LoRA 回答刚生成的 100 条样本。只将题目交给模型，不输入标准答案；保留原始回答，供后续严格评分。结果与 Validation、Test 分开保存。

In [ ]:
import json
import hashlib
from collections import Counter
from pathlib import Path

# 复用前面已加载的 Phase 3 LoRA 和推理函数，不重新训练。
assert "lora_predict" in globals(), "请先运行 LoRA 加载与 lora_predict 定义单元格。"
assert "adapter_dir" in globals(), "找不到 Phase 3 adapter 路径。"

challenge_path = Path(
    "/content/drive/MyDrive/FleetMind-LLM/phase4/phase4_challenge_100.jsonl"
)
assert challenge_path.is_file(), f"找不到困难样本：{challenge_path}"

# 固定样本版本，避免中途换了题目却继续写入旧预测文件。
challenge_sha = hashlib.sha256(challenge_path.read_bytes()).hexdigest()
assert challenge_sha == (
    "36cab650f9053c9d96137a5ca3f77ecbb87e2bf6b04080f0abb1602512ccac64"
), "困难样本文件与刚才检查的版本不同"

with challenge_path.open("r", encoding="utf-8") as file:
    challenge_rows = [json.loads(line) for line in file if line.strip()]

ids = [row["sample_id"] for row in challenge_rows]
assert len(ids) == len(set(ids)) == 100
assert Counter(row["task"] for row in challenge_rows) == {"A": 50, "B": 50}

save_dir = challenge_path.parent
prediction_path = save_dir / "phase4_lora_challenge_predictions.jsonl"
config_path = save_dir / "phase4_lora_challenge_config.json"

# 记录模型权重、样本版本及生成设置，确保中断续跑时配置一致。
weight_path = Path(adapter_dir) / "adapter_model.safetensors"
assert weight_path.is_file(), f"找不到 adapter 权重：{weight_path}"
config = {
    "model": "Qwen/Qwen3-0.6B",
    "adapter_sha256": hashlib.sha256(weight_path.read_bytes()).hexdigest(),
    "challenge_sha256": challenge_sha,
    "max_new_tokens": 384,
    "enable_thinking": False,
    "do_sample": False,
}

if config_path.exists():
    with config_path.open("r", encoding="utf-8") as file:
        assert json.load(file) == config, "已有预测使用的配置不同，请勿混合结果"
else:
    assert not prediction_path.exists(), "预测文件存在，但缺少对应配置文件"
    config_path.write_text(
        json.dumps(config, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

# 已保存的样本直接跳过；每条新预测立即写入 Drive。
completed = set()
if prediction_path.exists():
    with prediction_path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                sample_id = json.loads(line)["sample_id"]
                assert sample_id not in completed, f"重复预测：{sample_id}"
                completed.add(sample_id)

assert completed <= set(ids), "预测文件包含不属于本版困难样本的编号"
print(f"已有结果：{len(completed)}/100")

for sample in challenge_rows:
    if sample["sample_id"] in completed:
        continue

    # lora_predict 只读取 system 和 user，不读取 assistant 标准答案。
    answer, seconds = lora_predict(sample, max_new_tokens=384)
    row = {
        "sample_id": sample["sample_id"],
        "task": sample["task"],
        "prediction": answer,
        "latency_seconds": round(seconds, 4),
    }

    with prediction_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")
        file.flush()

    completed.add(sample["sample_id"])
    if len(completed) % 10 == 0:
        print(f"已完成：{len(completed)}/100")

print("预测完成；结果文件：", prediction_path)

### 困难样本结果

按原始回答严格评分：Task A 检查 JSON、结构、字段和 `null`；Task B 检查标签准确率，并单独查看时间边界与时间倒序题。困难样本是针对性补充检查，结果不代表正式 Test 成绩。

In [ ]:
import json
import re
from collections import Counter
from datetime import datetime
from pathlib import Path

save_dir = Path("/content/drive/MyDrive/FleetMind-LLM/phase4")
data_path = save_dir / "phase4_challenge_100.jsonl"
pred_path = save_dir / "phase4_lora_challenge_predictions.jsonl"

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

samples = read_jsonl(data_path)
rows = read_jsonl(pred_path)
predictions = {row["sample_id"]: row["prediction"] for row in rows}
assert len(samples) == len(rows) == len(predictions) == 100
assert {s["sample_id"] for s in samples} == set(predictions)

fields = [
    "load_id", "truck_id", "origin_city", "destination_city",
    "revenue_usd", "fuel_used_gallons",
    "scheduled_delivery", "actual_delivery",
]
text_fields = set(fields) - {"revenue_usd", "fuel_used_gallons"}

def strict_parse(raw):
    """直接解析原始回答；不删除代码围栏或修正内容。"""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None

def valid_a_schema(answer):
    """八个字段须齐全、无多余字段，非空值的类型须正确。"""
    if not isinstance(answer, dict) or set(answer) != set(fields):
        return False
    for field in fields:
        value = answer[field]
        if value is None:
            continue
        if field in text_fields and not isinstance(value, str):
            return False
        if field not in text_fields and (
            isinstance(value, bool) or not isinstance(value, (int, float))
        ):
            return False
    return True

a_count = Counter()
a_errors = []
b_count = Counter()
b_errors = []

for sample in samples:
    sample_id = sample["sample_id"]
    expected = json.loads(sample["messages"][2]["content"])
    answer = strict_parse(predictions[sample_id])
    question = sample["messages"][1]["content"]

    if sample["task"] == "A":
        a_count["total"] += 1
        a_count["json"] += isinstance(answer, dict)
        a_count["schema"] += valid_a_schema(answer)

        for field in fields:
            truth = expected[field]
            if truth is None:
                a_count["null_total"] += 1

            correct = (
                isinstance(answer, dict)
                and field in answer
                and type(answer[field]) is type(truth)
                and answer[field] == truth
            )
            a_count["fields_correct"] += correct
            if truth is None:
                a_count["null_correct"] += correct
            if not correct:
                a_errors.append((sample_id, field, truth,
                                 answer.get(field) if isinstance(answer, dict) else answer))
        continue

    # 与 Validation 的 Task B 口径一致：只接受单字段合法 JSON。
    truth = expected["delay_label"]
    predicted = (
        answer["delay_label"]
        if isinstance(answer, dict)
        and set(answer) == {"delay_label"}
        and answer["delay_label"] in {"delayed", "on_time"}
        else "无效"
    )
    b_count["total"] += 1
    b_count["correct"] += predicted == truth
    b_count["invalid"] += predicted == "无效"
    b_count[(truth, predicted)] += 1

    # 从明确的字段名提取时间，不假设计划时间一定写在前面。
    stamp = r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{6})"
    planned = re.search(r"计划时间：" + stamp, question)
    actual = re.search(r"实际时间：" + stamp, question)
    assert planned and actual
    plan_time = datetime.fromisoformat(planned.group(1))
    actual_time = datetime.fromisoformat(actual.group(1))
    delta = (actual_time - plan_time).total_seconds()

    if abs(delta) <= 1:
        b_count["boundary_total"] += 1
        b_count["boundary_correct"] += predicted == truth
    if question.index("实际时间：") < question.index("计划时间："):
        b_count["reversed_total"] += 1
        b_count["reversed_correct"] += predicted == truth

    if predicted != truth:
        b_errors.append((sample_id, truth, predicted, delta))

print("===== Task A：50 条 =====")
print(f"严格 JSON：{a_count['json']}/50")
print(f"结构正确：{a_count['schema']}/50")
print(f"字段正确：{a_count['fields_correct']}/400")
print(f"null 正确：{a_count['null_correct']}/{a_count['null_total']}")
print("错误字段示例（最多 5 个）：", a_errors[:5])

print("\n===== Task B：50 条 =====")
print(f"Accuracy：{b_count['correct']}/50 = {b_count['correct']/50:.2%}")
print(f"无效回答：{b_count['invalid']}")
print(f"时间差 ≤ 1 秒：{b_count['boundary_correct']}/{b_count['boundary_total']}")
print(f"实际时间先出现：{b_count['reversed_correct']}/{b_count['reversed_total']}")
print("混淆计数：")
for truth in ("delayed", "on_time"):
    print(truth, {
        predicted: b_count[(truth, predicted)]
        for predicted in ("delayed", "on_time", "无效")
    })
print("错误样本（编号、正确标签、预测、实际−计划秒数）：")
for error in b_errors:
    print(error)

### 困难样本结果与结论

| 检查项 | LoRA 结果 | 观察 |
|---|---:|---|
| Task A 严格 JSON / 结构 | 50/50 | 输出格式均正确 |
| Task A 字段 / `null` | 400/400；49/49 | 字段提取与缺失处理均正确 |
| Task B 总体 Accuracy | 45/50（90.00%） | 无无效回答 |
| Task B 时间差 ≤ 1 秒 | 20/25 | 5 条错误均发生在此组 |
| Task B 实际时间先出现 | 20/25 | 同一批 5 条错误也属于此组 |

**错误特点**：5 条错误均是“实际时间比计划时间晚 **1 微秒**”，正确标签为 `delayed`，模型却输出 `on_time`。它们是同一条件在 5 个不同日期上的重复。由于“1 微秒延误”和“实际时间先出现”同时存在，当前结果**无法单独判断**是哪一个因素导致错误。

**结论**：LoRA 在这组人工构造的 Task A 样本上全部正确；Task B 暴露出微秒级延误判断的弱点。这是针对性补充检查，不能把 90.00% 当作正式 Test 成绩，也不能直接与 Validation 的 99.40% 比较总体泛化能力。

## Phase 4.5｜错误分析与模型局限

### 4.5.1｜选取代表性案例

从已保存的 Validation 预测中，按固定规则选取 LoRA 的错误、Few-shot 的两类误判及 Task A 的格式或缺失字段错误；另列出 1 条人工压力测试案例。案例用于说明错误类型，不用于调参，也不代替总体指标。

In [ ]:
import json
from pathlib import Path
from collections import Counter

phase4_dir = Path("/content/drive/MyDrive/FleetMind-LLM/phase4")
paths = {
    "zero": Path("/content/phase4_zero_shot_val_predictions.jsonl"),
    "few": Path("/content/phase4_few_shot_val_predictions.jsonl"),
    "lora": phase4_dir / "phase4_lora_val_predictions.jsonl",
    "challenge": phase4_dir / "phase4_lora_challenge_predictions.jsonl",
}
for name, path in paths.items():
    assert path.is_file(), f"缺少 {name} 预测文件：{path}"
assert "val_samples" in globals(), "请先加载扩展 Validation"

def load_predictions(path):
    """按 sample_id 读取已有回答，不依赖文件行顺序。"""
    with path.open("r", encoding="utf-8") as file:
        rows = [json.loads(line) for line in file if line.strip()]
    result = {row["sample_id"]: row["prediction"] for row in rows}
    assert len(rows) == len(result), f"预测编号重复：{path}"
    return result

pred = {name: load_predictions(path) for name, path in paths.items()}
validation = {s["sample_id"]: s for s in val_samples}
assert len(validation) == 1000
assert all(set(validation) == set(pred[name]) for name in ("zero", "few", "lora"))

def parse(raw):
    """严格解析原始回答；格式错误保持为无效。"""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None

def b_label(raw):
    answer = parse(raw)
    if isinstance(answer, dict) and set(answer) == {"delay_label"}:
        return answer["delay_label"]
    return "无效"

cases = []
used_ids = set()

def add_case(sample_id, category, detail):
    """避免同一道题在案例清单中重复出现。"""
    if sample_id not in used_ids:
        cases.append((category, sample_id, detail))
        used_ids.add(sample_id)

# 1. 收录 LoRA 在 Validation 的全部 3 条 Task B 错误。
for sample_id in sorted(validation):
    sample = validation[sample_id]
    if sample["task"] != "B":
        continue
    truth = json.loads(sample["messages"][-1]["content"])["delay_label"]
    answer = b_label(pred["lora"][sample_id])
    if answer != truth:
        add_case(sample_id, "LoRA 时间判断", f"正确 {truth}；预测 {answer}")

# 2. Few-shot 的两类误判各取前 4 条，编号排序保证可复现。
for truth, category in [
    ("delayed", "Few-shot 漏报延误"),
    ("on_time", "Few-shot 误报延误"),
]:
    selected = 0
    for sample_id in sorted(validation):
        sample = validation[sample_id]
        if sample["task"] != "B" or sample_id in used_ids:
            continue
        expected = json.loads(sample["messages"][-1]["content"])["delay_label"]
        answer = b_label(pred["few"][sample_id])
        if expected == truth and answer != truth:
            add_case(sample_id, category, f"正确 {truth}；预测 {answer}")
            selected += 1
            if selected == 4:
                break
    assert selected == 4, f"{category} 案例不足"

# 3. Few-shot 中应输出 null 却答错的 Task A，取前 2 条。
selected = 0
for sample_id in sorted(validation):
    sample = validation[sample_id]
    if sample["task"] != "A":
        continue
    expected = json.loads(sample["messages"][-1]["content"])
    answer = parse(pred["few"][sample_id])
    wrong_null = [
        field for field, value in expected.items()
        if value is None and (
            not isinstance(answer, dict)
            or field not in answer
            or answer[field] is not None
        )
    ]
    if wrong_null:
        add_case(sample_id, "Few-shot 缺失字段", f"应为 null：{wrong_null[0]}")
        selected += 1
        if selected == 2:
            break
assert selected == 2

# 4. Zero-shot 中原始回答不是合法 JSON 的 Task A，取前 2 条。
selected = 0
for sample_id in sorted(validation):
    if validation[sample_id]["task"] == "A" and parse(pred["zero"][sample_id]) is None:
        add_case(sample_id, "Zero-shot 格式错误", "原始回答无法严格解析为 JSON")
        selected += 1
        if selected == 2:
            break
assert selected == 2

# 5. 人工压力测试只取一条微秒级案例，与 Validation 案例分开标记。
challenge_id = "CHALLENGE_B_007"
assert challenge_id in pred["challenge"]
add_case(challenge_id, "人工压力测试", "晚 1 微秒；LoRA 预测 on_time")

print("代表性案例总数：", len(cases))
print("类别数量：", dict(Counter(category for category, _, _ in cases)))
for category, sample_id, detail in cases:
    print(f"{category} | {sample_id} | {detail}")

### 4.5.2｜错误类型与模型局限

共选取 16 条不重复的代表性案例，其中 15 条来自 Validation，1 条来自人工压力测试。

| 方法与错误类型 | 案例数 | 观察 |
|---|---:|---|
| LoRA：延误误报 | 3 | 均将 `on_time` 判为 `delayed`；包括亚秒级和明显提前的交付 |
| Few-shot：漏报延误 | 4 | 正确标签为 `delayed`，却预测 `on_time` |
| Few-shot：误报延误 | 4 | 正确标签为 `on_time`，却预测 `delayed` |
| Few-shot：缺失字段 | 2 | 两例都未正确处理应为 `null` 的 `scheduled_delivery` |
| Zero-shot：输出格式 | 2 | 原始回答无法按严格 JSON 解析 |
| LoRA：人工压力测试 | 1 | 晚 1 微秒，却预测为 `on_time`；该条件在压力测试中重复出现 5 次 |

**阶段结论**：LoRA 在统一 Validation 上表现最好，但 Task B 仍有 3 条误报，微秒级压力测试也暴露出边界判断弱点。Few-shot 改善了 Task A 输出格式，却同时出现延误漏报与误报。Zero-shot 的主要格式问题会显著拉低其严格 JSON 指标。

这些案例展示了**错误表现**，不能仅凭案例确定内部原因；人工压力测试结果也不作为正式 Test 成绩。Phase 4 到此结束，后续进入 Phase 5 的训练规模实验，最终模型选定后才使用封存 Test。

## Phase 4｜阶段总结

**目标**：扩展并核对评测数据，在同一份 Validation 上比较方法，分析错误，并用独立构造的题目检查模型弱点。本阶段没有重新训练模型。

### 1. 数据与质量检查

| 数据集 | 指令数 | Task A / B | 用途 |
|---|---:|---:|---|
| Train | 1,600 | 800 / 800 | Phase 3 已用于训练 |
| Validation | 1,000 | 500 / 500 | Phase 4 方法对比与错误分析 |
| Test | 2,000 | 1,000 / 1,000 | 封存，留待 Phase 5 选定最终模型后评测 |

扩展数据保留原有样本，并检查了编号、A/B 配对、数据划分、标签和集合重叠。Test 数据包的 SHA-256、数量与编号已核对；**尚未生成或评分 Test 预测**。

### 2. Validation 方法对比

| 方法 | Task A 严格 JSON | Task A 字段正确 | Task B Accuracy | Task B Macro-F1 | `on_time` Recall |
|---|---:|---:|---:|---:|---:|
| 规则方法 | 不适用 | 不适用 | 100% | 100% | 100% |
| Zero-shot | 0.20% | 7/4000（0.18%） | 66.80% | 40.05% | 0% |
| Few-shot | 100% | 3963/4000（99.08%） | 62.20% | 54.01% | 30.30% |
| Phase 3 LoRA | **100%** | **4000/4000（100%）** | **99.40%** | **99.32%** | **98.18%** |

规则方法只完成 Task B，是确定性参考，不属于语言模型成绩。Zero-shot 的 Task A 严格分数受到代码围栏影响；仅去掉完整围栏后，字段正确率为 **3759/4000（93.97%）**，但这不替代原始输出的严格评分。

### 3. 错误分析

- **LoRA Task B**：500 条中错 3 条，全部把 `on_time` 判为 `delayed`。其中 1 条提前不足 1 秒，另 2 条分别提前约 1 小时和 2 小时，不能把所有错误归因于时间边界。
- **Few-shot Task B**：相对 Zero-shot，修正了 51 条错题，也使 74 条原本正确的题变错，净减少 23 条正确答案。
- **Few-shot Task A**：37 次字段错误中，36 次涉及应输出 `null` 的字段。
- **标签复核**：Validation 的 500 条 Task B 均可解析时间，重新计算的标签与标准答案全部一致。
- **代表性案例**：按固定规则选取了 16 条案例，用于展示格式、缺失字段和延误误判等错误表现，不把案例数量当作总体错误率。

### 4. 人工困难样本

另构造 100 条题目（Task A/B 各 50 条），与正式 Validation 和 Test 分开报告。

| 指标 | LoRA 结果 |
|---|---:|
| Task A 严格 JSON / 结构 | 50/50 |
| Task A 字段 / `null` | 400/400；49/49 |
| Task B Accuracy | 45/50（90%） |

Task B 的 5 条错误都是“实际交付晚 **1 微秒**，却预测为 `on_time`”。这实际上是同一条件在不同日期上的重复；它也同时采用了“实际时间先出现”的表述，因此目前不能单独确定是哪一个因素导致错误。这组题目是在观察 Validation 错误后设计的**探索性压力测试**，其 90% 不代表正式泛化成绩。

### 5. 阶段结论与下一步

Phase 3 LoRA 在统一 Validation 上同时完成了 Task A 和 Task B，Task A 严格字段全部正确，Task B Accuracy 为 99.40%。它仍有时间判断错误，尤其需要如实记录微秒级压力测试中的表现。规则方法在 Task B 达到 100%，应作为清晰的业务参考。

**下一步进入 Phase 5**：开展训练数据规模等实验，使用相同的 Validation 指标选择模型。现有 Phase 3 adapter 的实际 LoRA Rank 为 **16**；如果 Phase 5 按大纲使用 Rank **8**，必须作为新的训练配置单独记录。选定最终模型后，才对封存 Test 进行最终评测。